# NBA Scout Data Processing

This notebook builds the exploratory data layer for NBA scouting, player similarity, short-term production forecasting, long-term player trajectory forecasting, and salary analysis.

Primary gold outputs:

- `player_role_features_clean.parquet`: cleaned player-season role profiles for similarity search and candidate retrieval.
- `performance_training_clean.parquet`: point-in-time rolling form features with future five-game production targets.
- `salary_training_clean.parquet`: cleaned player-season salary table with role, production, bio, and salary-cap context.
- `long_term_player_forecast_training.parquet`: season-anchor table for multi-season availability and production forecasting.

The notebook is structured as an experimentation pipeline: raw and silver inputs are normalized, clean modeling tables are produced, and baseline/selected models are evaluated with temporal splits and MLflow tracking.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, f1_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [ ]:
# Colab/bootstrap install. nba-api import path is nba_api.
%pip install -q nba-api kagglehub mlflow lightgbm


## 1. Paths and Input Contracts

The notebook resolves data from Google Drive by default: `My Drive/nba-scout-assistant/data`.

In Colab, Google Drive is mounted before paths are resolved. The expected data root is:

```text
/content/drive/MyDrive/nba-scout-assistant/data
```

For local runs, set `NBA_SCOUT_DATA_DIR` to the project `data` directory when the default path is not correct.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DRIVE_PROJECT_FOLDER_ID = "1aweR7CoNntwybx3Y9eTAXje2AfV_8K8H"
COLAB_DRIVE_DATA_DIR = Path("/content/drive/MyDrive/nba-scout-assistant/data")
COLAB_LEGACY_DRIVE_DATA_DIR = Path("/content/drive/My Drive/nba-scout-assistant/data")

def running_in_colab() -> bool:
    """Input: none. Output: True when the notebook is running in Google Colab."""
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False

def mount_google_drive_if_needed() -> None:
    """Input: none. Output: mounted /content/drive when running in Colab and Drive data is not visible yet."""
    if not running_in_colab():
        return
    if COLAB_DRIVE_DATA_DIR.exists() or COLAB_LEGACY_DRIVE_DATA_DIR.exists():
        return
    from google.colab import drive  # type: ignore
    print("Mounting Google Drive so the project data folder is visible...")
    drive.mount("/content/drive")

def first_existing_path(paths: list[Path]) -> Path | None:
    """Input: candidate paths. Output: first existing Path or None. Used to locate Drive/local data folder."""
    for path in paths:
        if path.exists():
            return path
    return None

mount_google_drive_if_needed()

env_data_dir = os.getenv("NBA_SCOUT_DATA_DIR")
drive_data_candidates = [
    COLAB_DRIVE_DATA_DIR,
    COLAB_LEGACY_DRIVE_DATA_DIR,
    Path.home() / "Google Drive" / "My Drive" / "nba-scout-assistant" / "data",
    Path.home() / "Library" / "CloudStorage" / "GoogleDrive-MyDrive" / "nba-scout-assistant" / "data",
]

DATA_DIR = Path(env_data_dir).expanduser().resolve() if env_data_dir else first_existing_path(drive_data_candidates)
if DATA_DIR is None and running_in_colab() and env_data_dir is None:
    raise FileNotFoundError(
        "Google Drive is mounted, but the project data folder was not found. "
        f"Expected: {COLAB_DRIVE_DATA_DIR}. "
        f"Drive folder ID checked from connector: {DRIVE_PROJECT_FOLDER_ID}. "
        "Make sure the folder is under My Drive, not only Shared with me, or set NBA_SCOUT_DATA_DIR."
    )
elif DATA_DIR is None:
    DATA_DIR = PROJECT_ROOT / "data"
    print(f"Drive data folder not found. Falling back to local data folder: {DATA_DIR}")
    print("Expected Colab Drive path:", COLAB_DRIVE_DATA_DIR)
else:
    print(f"Using data folder: {DATA_DIR}")

BRONZE_DIR = DATA_DIR / "bronze"
RAW_DIR = DATA_DIR / "raw"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
GOLD_DIR.mkdir(parents=True, exist_ok=True)

PLAYERS_PATH = RAW_DIR / "players.parquet"
PLAYER_BIO_PATH = RAW_DIR / "player_bio" / "eoin_players.csv"
NBA_TRADITIONAL_DIR = RAW_DIR / "nba_traditional"
GAME_LOGS_PATH = SILVER_DIR / "player_game_logs.parquet"
NBA_API_GAME_LOGS_PATH = RAW_DIR / "player_game_logs_nba_api.parquet"
SEASON_STATS_PATH = RAW_DIR / "player_season_stats.parquet"
ADVANCED_STATS_PATH = RAW_DIR / "player_stats_advanced" / "player_stats_advanced_rs.csv"
USAGE_STATS_PATH = RAW_DIR / "player_stats_advanced" / "player_stats_usage_rs.csv"
DEFENSE_STATS_PATH = RAW_DIR / "player_stats_advanced" / "player_stats_defense_rs.csv"
ADVANCED_PATCH_2023_DIR = RAW_DIR / "player_stats_advanced_patch" / "2023_24"
ADVANCED_PATCH_2024_DIR = RAW_DIR / "player_stats_advanced_patch" / "2024_25"
RODNEY_ADVANCED_PATH = ADVANCED_PATCH_2023_DIR / "Advanced.csv"
RATIN_ADVANCED_2024_25_PATH = ADVANCED_PATCH_2024_DIR / "NBA Player Advanced Stats_2024-25.csv"
RATIN_TOTAL_2024_25_PATH = ADVANCED_PATCH_2024_DIR / "NBA Player Stats_2024-25_Total.csv"
SALARY_CAP_PATH = RAW_DIR / "salary_cap" / "salary_cap_by_season.csv"
PLAYER_SEASON_SALARIES_PATH = SILVER_DIR / "player_season_salaries.parquet"

OUTPUT_ROLE_FEATURES = GOLD_DIR / "player_role_features.parquet"
OUTPUT_PERFORMANCE_TRAINING = GOLD_DIR / "performance_training.parquet"
OUTPUT_SALARY_TRAINING = GOLD_DIR / "salary_training.parquet"
OUTPUT_ROLE_FEATURES_CLEAN = GOLD_DIR / "player_role_features_clean.parquet"
OUTPUT_PERFORMANCE_TRAINING_CLEAN = GOLD_DIR / "performance_training_clean.parquet"
OUTPUT_SALARY_TRAINING_CLEAN = GOLD_DIR / "salary_training_clean.parquet"

MLFLOW_BACKEND_DB = Path(os.getenv("MLFLOW_BACKEND_DB", str(DATA_DIR.parent / "mlflow.db"))).expanduser().resolve()
MLFLOW_ARTIFACT_DIR = Path(os.getenv("MLFLOW_ARTIFACT_DIR", str(DATA_DIR.parent / "mlartifacts"))).expanduser().resolve()
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "nba_scout_performance_forecasting")
MLFLOW_BACKEND_DB.parent.mkdir(parents=True, exist_ok=True)
MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"MLflow backend DB: {MLFLOW_BACKEND_DB}")
print(f"MLflow artifact dir: {MLFLOW_ARTIFACT_DIR}")

critical_input_paths = {
    "players parquet": PLAYERS_PATH,
    "player bio csv": PLAYER_BIO_PATH,
    "season stats parquet": SEASON_STATS_PATH,
    "salary cap csv": SALARY_CAP_PATH,
    "2023-24 advanced patch": RODNEY_ADVANCED_PATH,
    "2024-25 advanced patch": RATIN_ADVANCED_2024_25_PATH,
    "salary parquet": PLAYER_SEASON_SALARIES_PATH,
    "game logs parquet": GAME_LOGS_PATH,
}
for label, path in critical_input_paths.items():
    print(f"{label}: {'FOUND' if path.exists() else 'missing'} - {path}")

RAW_DIR, GOLD_DIR


In [ ]:
RAW_SCHEMAS = {
    "players": [
        "player_id", "player_name", "birth_date", "position", "height", "weight",
    ],
    "player_game_logs": [
        "player_id", "game_date", "game_id", "season", "team_id", "minutes", "points", "assists",
        "rebounds", "offensive_rebounds", "defensive_rebounds", "steals", "blocks", "personal_fouls",
        "turnovers", "true_shooting_pct", "opponent", "home_away", "rest_days",
    ],
    "player_season_stats": [
        "player_id", "season", "team_id", "age", "minutes", "usage_pct", "points_per_100",
        "assists_per_100", "rebounds_per_100", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate",
        "foul_rate", "pace", "possessions", "offensive_rating", "defensive_rating",
    ],
    "salary_cap_by_season": ["season", "salary_cap_usd", "tax_level_usd"],
    "player_season_salaries": [
        "player_name", "team", "season_start_year", "season_end_year", "season_label", "salary_usd",
        "source", "source_file", "collected_at",
    ],
}

def read_table(path: Path) -> pd.DataFrame:
    """Input: CSV/parquet path. Output: DataFrame, or empty DataFrame if file is missing."""
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

def report_schema(name: str, df: pd.DataFrame, expected_columns: list[str]) -> None:
    """Input: table name, DataFrame, expected columns. Output: printed schema/missing-column report."""
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    missing = sorted(set(expected_columns) - set(df.columns))
    extra = sorted(set(df.columns) - set(expected_columns))
    if missing:
        print("  Missing expected columns:", missing)
    if extra:
        print("  Extra columns:", extra[:25], "..." if len(extra) > 25 else "")


## 2. Load Raw Tables

## 2A. Historical Game Logs

Historical player game logs are sourced from the Kaggle dataset `szymonjwiak/nba-traditional` and normalized into `data/silver/player_game_logs.parquet`.

The loader does not depend on a fixed file name. It scans tabular files, identifies player box-score candidates using core columns such as points, assists, and rebounds, then maps the selected file into a canonical schema.

In [ ]:
try:
    import kagglehub
    KAGGLEHUB_AVAILABLE = True
except ImportError:
    KAGGLEHUB_AVAILABLE = False
    print("kagglehub is not installed. Run `%pip install kagglehub` in this notebook if you want to fetch Kaggle datasets.")

# nba_api remains available as a fallback for lightweight metadata, not as the primary game-log source.
# Nếu kernel chưa có nba_api, chạy cell install `nba-api` ở đầu notebook rồi restart kernel nếu cần.

try:
    from nba_api.stats.endpoints import leaguedashplayerstats, playergamelogs
    from nba_api.stats.static import players as nba_static_players
    NBA_API_AVAILABLE = True
except ImportError:
    NBA_API_AVAILABLE = False
    print("nba_api is not installed. Run `%pip install nba-api` in this notebook if you want to fetch data.")


In [ ]:
AUTO_FETCH_KAGGLE_TRADITIONAL_IF_MISSING = True
AUTO_FETCH_NBA_API_IF_MISSING = False
OVERWRITE_EXISTING_NBA_API_RAW = False
KAGGLE_TRADITIONAL_DATASET = "szymonjwiak/nba-traditional"
KAGGLE_ADVANCED_2023_DATASET = "rodneycarroll78/nba-stats-1980-2024"
KAGGLE_ADVANCED_2024_DATASET = "ratin21/nba-player-stats-2024-25-per-game"
AUTO_FETCH_ADVANCED_PATCH_IF_MISSING = True

# NBA season format used by stats.nba.com. Keep this aligned with salary data availability.
FETCH_SEASONS = [
    "2016-17", "2017-18", "2018-19", "2019-20", "2020-21",
    "2021-22", "2022-23", "2023-24", "2024-25",
]

# Temporal split for short-term and salary experiments. Do not random-split these datasets.
TRAIN_END_SEASON = "2022-23"
VALIDATION_SEASONS = ["2023-24"]
TEST_SEASONS = ["2024-25"]

# Long-term horizons need complete future labels through h3. With data through 2024-25,
# the latest valid long-term test anchor is 2021-22 because 2021-22 + h3 = 2024-25.
LONG_TERM_TRAIN_END_SEASON = "2019-20"
LONG_TERM_VALIDATION_SEASONS = ["2020-21"]
LONG_TERM_TEST_SEASONS = ["2021-22"]
SEASON_TYPE = "Regular Season"
NBA_API_TIMEOUT_SECONDS = 180
NBA_API_MAX_RETRIES = 3

# NBA Stats can throttle requests. Increase this if calls fail intermittently.
REQUEST_SLEEP_SECONDS = 3.0
NBA_API_CACHE_DIR = RAW_DIR / "nba_api_cache"
NBA_API_CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import re

def to_snake_case(column: str) -> str:
    """Input: raw column name. Output: normalized snake_case column name for schema matching."""
    column = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", str(column))
    column = re.sub(r"[^a-zA-Z0-9]+", "_", column)
    return column.strip("_").lower()

def read_tabular_file(path: Path) -> pd.DataFrame:
    """Input: CSV/parquet file path. Output: DataFrame read with the appropriate pandas reader."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {path}")

def download_nba_traditional_dataset() -> Path:
    """Input: none. Output: local KaggleHub dataset path for szymonjwiak/nba-traditional."""
    if not KAGGLEHUB_AVAILABLE:
        raise ImportError("Install kagglehub first by running `%pip install kagglehub`, then restart/re-run the notebook.")
    NBA_TRADITIONAL_DIR.mkdir(parents=True, exist_ok=True)
    dataset_path = kagglehub.dataset_download(
        KAGGLE_TRADITIONAL_DATASET,
        output_dir=str(NBA_TRADITIONAL_DIR),
    )
    print("Dataset downloaded to:", dataset_path)
    return Path(dataset_path)

def download_kaggle_dataset(dataset_slug: str, output_dir: Path) -> Path:
    """Input: KaggleHub dataset slug and output dir. Output: downloaded dataset path."""
    if not KAGGLEHUB_AVAILABLE:
        raise ImportError("Install kagglehub first by running `%pip install kagglehub`, then restart/re-run the notebook.")
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset_path = kagglehub.dataset_download(dataset_slug, output_dir=str(output_dir))
    print("Dataset downloaded to:", dataset_path)
    return Path(dataset_path)

def normalize_name_key(value: object) -> str | None:
    """Input: player name. Output: normalized key for cross-source player joins."""
    if pd.isna(value):
        return None
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def normalize_team_abbreviation(value: object) -> str | None:
    """Input: source team abbreviation. Output: abbreviation aligned with game-log team ids."""
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    mapping = {"CHO": "CHA", "BRK": "BKN", "PHO": "PHX", "NOH": "NOP", "NOK": "NOP", "NJN": "BKN"}
    return mapping.get(text, text)

def percent_to_ratio(values: pd.Series) -> pd.Series:
    """Input: percentage-like Series. Output: ratio scale where values above 1.5 are divided by 100."""
    numeric = pd.to_numeric(values, errors="coerce")
    return numeric.where(numeric <= 1.5, numeric / 100)

def season_label_from_bref_end_year(value: object) -> str | None:
    """Input: Basketball-Reference season end year. Output: season label such as 2023-24."""
    if pd.isna(value):
        return None
    end_year = int(value)
    return f"{end_year - 1}-{str(end_year)[-2:]}"

def find_tabular_files(data_dir: Path) -> list[Path]:
    """Input: dataset directory. Output: all CSV/parquet files found recursively."""
    return sorted(path for path in data_dir.rglob("*") if path.suffix.lower() in {".csv", ".parquet"})

def find_player_boxscore_file(data_dir: Path) -> Path:
    """Input: raw dataset directory. Output: most likely player-level box-score file path."""
    signal_groups = [
        {"pts", "ast", "reb"},
        {"points", "assists", "rebounds_total"},
        {"points", "assists", "rebounds"},
    ]
    candidates = []
    for path in find_tabular_files(data_dir):
        try:
            sample = read_tabular_file(path)
        except Exception as exc:
            print(f"Could not read {path}: {exc}")
            continue
        normalized_columns = {to_snake_case(column) for column in sample.columns}
        best_match = max((len(group & normalized_columns) for group in signal_groups), default=0)
        player_signals = {"player_id", "playerid", "person_id", "person_name", "player_name", "player"}
        team_only_signals = {"teamid", "team_id"}
        has_player_signal = bool(player_signals & normalized_columns)
        team_only_score = len(team_only_signals & normalized_columns)
        if best_match >= 2 and has_player_signal:
            candidates.append({
                "path": path,
                "shape": sample.shape,
                "columns": sample.columns.tolist(),
                "score": best_match + 2 * int(has_player_signal) - team_only_score,
            })

    for candidate in candidates:
        print("Candidate:", candidate["path"], candidate["shape"])
        print("Columns:", candidate["columns"])
    if not candidates:
        raise FileNotFoundError("No player box-score candidate found in nba-traditional dataset.")
    return sorted(candidates, key=lambda item: (item["score"], item["shape"][0]), reverse=True)[0]["path"]

def season_end_year_from_date(game_date: pd.Timestamp) -> int | None:
    """Input: game date. Output: NBA season end year, e.g. 2016-17 -> 2017."""
    if pd.isna(game_date):
        return None
    return game_date.year + 1 if game_date.month >= 7 else game_date.year

def season_label_from_end_year(end_year: object) -> str | None:
    """Input: season end year. Output: season label such as 2024-25."""
    if pd.isna(end_year):
        return None
    end_year = int(end_year)
    start_year = end_year - 1
    return f"{start_year}-{str(end_year)[-2:]}"

def parse_minutes(value: object) -> float | None:
    """Input: minutes value as number or MM:SS text. Output: minutes as float."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    if ":" in text:
        minutes, seconds = text.split(":", 1)
        return float(minutes) + float(seconds) / 60
    return float(text)

def canonicalize_kaggle_player_game_logs(raw: pd.DataFrame) -> pd.DataFrame:
    """Input: raw Kaggle player box-score DataFrame. Output: canonical silver player_game_logs DataFrame."""
    df = raw.copy()
    df.columns = [to_snake_case(column) for column in df.columns]
    rename_map = {
        "gameid": "game_id",
        "date": "game_date",
        "playerid": "player_id",
        "player": "player_name",
        "person_id": "player_id",
        "person_name": "player_name",
        "game_id": "game_id",
        "game_date": "game_date",
        "team_id": "team_id",
        "team": "team_abbreviation",
        "team_tricode": "team_abbreviation",
        "team_abbreviation": "team_abbreviation",
        "min": "minutes",
        "minutes": "minutes",
        "pts": "points",
        "points": "points",
        "ast": "assists",
        "assists": "assists",
        "reb": "rebounds",
        "rebounds_total": "rebounds",
        "oreb": "offensive_rebounds",
        "dreb": "defensive_rebounds",
        "stl": "steals",
        "blk": "blocks",
        "pf": "personal_fouls",
        "home": "home_team",
        "away": "away_team",
        "tov": "turnovers",
        "turnovers": "turnovers",
        "fgm": "fgm",
        "fga": "fga",
        "3_pm": "fg3m",
        "3_pa": "fg3a",
        "3pm": "fg3m",
        "3pa": "fg3a",
        "fg3m": "fg3m",
        "fg3a": "fg3a",
        "ftm": "ftm",
        "fta": "fta",
    }
    df = df.rename(columns={old: new for old, new in rename_map.items() if old in df.columns})
    if "team_id" not in df.columns and "team_abbreviation" in df.columns:
        df["team_id"] = df["team_abbreviation"]
    required = ["player_id", "game_id", "game_date", "team_id", "minutes", "points", "assists", "rebounds"]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required player game-log columns: {missing}. Available columns: {df.columns.tolist()}")

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df["minutes"] = df["minutes"].map(parse_minutes)
    numeric_columns = [
        "points", "assists", "rebounds", "offensive_rebounds", "defensive_rebounds",
        "steals", "blocks", "personal_fouls", "turnovers", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta",
    ]
    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    if {"team_abbreviation", "home_team", "away_team"}.issubset(df.columns):
        is_home = df["team_abbreviation"].astype(str).eq(df["home_team"].astype(str))
        df["home_away"] = np.where(is_home, "HOME", "AWAY")
        df["opponent"] = np.where(is_home, df["away_team"], df["home_team"])

    if {"points", "fga", "fta"}.issubset(df.columns):
        denominator = 2 * (df["fga"] + 0.44 * df["fta"]).replace(0, np.nan)
        df["true_shooting_pct"] = df["points"] / denominator

    if "season_end_year" not in df.columns:
        df["season_end_year"] = df["game_date"].map(season_end_year_from_date).astype("Int64")
    df["season_label"] = df["season_end_year"].map(season_label_from_end_year)
    df["season"] = df["season_label"]

    season_type_cols = [column for column in df.columns if "season" in column or "type" in column]
    print("Season/type columns:", season_type_cols)
    for column in season_type_cols:
        if column != "season" and df[column].dtype == "object":
            values = df[column].astype(str).str.lower()
            if values.str.contains("regular").any():
                before = len(df)
                df = df[values.str.contains("regular", na=False)].copy()
                print(f"Filtered regular season using {column}: {before:,} -> {len(df):,}")
                break

    df = df[df["season_end_year"].between(2017, 2025)].copy()
    duplicate_key = ["player_id", "game_id", "team_id"]
    print("Duplicate rows:", df.duplicated(subset=duplicate_key, keep=False).sum())
    display(df[required].isna().mean().sort_values(ascending=False).to_frame("missing_rate"))

    keep_cols = [
        "player_id", "player_name", "game_date", "game_id", "season", "season_end_year", "season_label",
        "team_id", "team_abbreviation", "opponent", "home_away", "minutes", "points", "assists",
        "rebounds", "offensive_rebounds", "defensive_rebounds", "steals", "blocks", "personal_fouls",
        "turnovers", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta", "true_shooting_pct",
    ]
    keep_cols = [column for column in keep_cols if column in df.columns]
    canonical = df[keep_cols].sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    canonical["rest_days"] = canonical.groupby("player_id")["game_date"].diff().dt.days
    return canonical

def build_players_from_game_logs(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical game logs. Output: minimal players table with placeholder bio fields."""
    if game_logs_df.empty or "player_name" not in game_logs_df.columns:
        return pd.DataFrame()
    players_df = game_logs_df[["player_id", "player_name"]].dropna().drop_duplicates("player_id").copy()
    players_df["birth_date"] = pd.NaT
    players_df["position"] = pd.NA
    players_df["height"] = pd.NA
    players_df["weight"] = pd.NA
    return players_df[["player_id", "player_name", "birth_date", "position", "height", "weight"]]

def position_from_bio_flags(row: pd.Series) -> str | None:
    """Input: one eoin Players.csv row. Output: compact position string derived from guard/forward/center flags."""
    positions = []
    if pd.to_numeric(row.get("guard"), errors="coerce") == 1:
        positions.append("G")
    if pd.to_numeric(row.get("forward"), errors="coerce") == 1:
        positions.append("F")
    if pd.to_numeric(row.get("center"), errors="coerce") == 1:
        positions.append("C")
    return "/".join(positions) if positions else None

def normalize_eoin_players(path: Path) -> pd.DataFrame:
    """Input: eoin Players.csv path. Output: canonical players table with birth date, position, height, and weight."""
    raw = pd.read_csv(path, low_memory=False)
    players_df = pd.DataFrame({
        "player_id": pd.to_numeric(raw["personId"], errors="coerce").astype("Int64"),
        "player_name": (
            raw["firstName"].fillna("").astype(str).str.strip()
            + " "
            + raw["lastName"].fillna("").astype(str).str.strip()
        ).str.strip(),
        "birth_date": pd.to_datetime(raw["birthDate"], errors="coerce"),
        "position": raw.apply(position_from_bio_flags, axis=1),
        "height": pd.to_numeric(raw["heightInches"], errors="coerce"),
        "weight": pd.to_numeric(raw["bodyWeightLbs"], errors="coerce"),
    })
    return players_df.dropna(subset=["player_id"]).drop_duplicates("player_id").reset_index(drop=True)

def needs_player_bio_refresh(path: Path) -> bool:
    """Input: canonical players path. Output: True when bio source should refresh missing profile columns."""
    if should_fetch(path):
        return True
    current = read_table(path)
    if current.empty:
        return True
    required_profile_cols = ["birth_date", "position", "height", "weight"]
    missing_cols = [column for column in required_profile_cols if column not in current.columns]
    if missing_cols:
        return True
    return current[required_profile_cols].isna().mean().mean() > 0.25

def build_season_stats_from_game_logs(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical game logs. Output: season-level baseline stats for role features."""
    if game_logs_df.empty:
        return pd.DataFrame()
    df = game_logs_df.copy()
    grouped = df.groupby(["player_id", "player_name", "season", "team_id"], dropna=False)
    agg_spec = {
        "games": ("game_id", "nunique"),
        "minutes": ("minutes", "sum"),
        "points": ("points", "sum"),
        "assists": ("assists", "sum"),
        "rebounds": ("rebounds", "sum"),
    }
    for optional_column in [
        "turnovers", "fga", "fg3a", "fta", "steals", "blocks", "defensive_rebounds", "personal_fouls",
    ]:
        if optional_column in df.columns:
            agg_spec[optional_column] = (optional_column, "sum")
    season = grouped.agg(**agg_spec).reset_index()
    for optional_column in [
        "turnovers", "fga", "fg3a", "fta", "steals", "blocks", "defensive_rebounds", "personal_fouls",
    ]:
        if optional_column not in season.columns:
            season[optional_column] = np.nan
    per_100_factor = 100 / season["minutes"].replace(0, np.nan)
    season["age"] = pd.NA
    season["usage_pct"] = pd.NA
    season["points_per_100"] = season["points"] * per_100_factor
    season["assists_per_100"] = season["assists"] * per_100_factor
    season["rebounds_per_100"] = season["rebounds"] * per_100_factor
    denom = 2 * (season["fga"] + 0.44 * season["fta"]).replace(0, np.nan)
    season["true_shooting_pct"] = season["points"] / denom
    season["three_point_attempt_rate"] = season["fg3a"] / season["fga"].replace(0, np.nan)
    season["free_throw_rate"] = season["fta"] / season["fga"].replace(0, np.nan)
    season["turnover_rate"] = season["turnovers"] / (season["fga"] + 0.44 * season["fta"] + season["turnovers"]).replace(0, np.nan)
    per_36_factor = 36 / season["minutes"].replace(0, np.nan)
    season["steal_rate"] = season["steals"] * per_36_factor
    season["block_rate"] = season["blocks"] * per_36_factor
    season["defensive_rebound_rate"] = season["defensive_rebounds"] * per_36_factor
    season["foul_rate"] = season["personal_fouls"] * per_36_factor
    season["pace"] = pd.NA
    season["possessions"] = pd.NA
    season["offensive_rating"] = pd.NA
    season["defensive_rating"] = pd.NA
    return season[["player_id", "player_name", "season", "team_id", "age", "minutes", "usage_pct", "points_per_100", "assists_per_100", "rebounds_per_100", "true_shooting_pct", "three_point_attempt_rate", "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate", "foul_rate", "pace", "possessions", "offensive_rating", "defensive_rating"]]

def normalize_thomas_player_season_stats(
    advanced_path: Path,
    usage_path: Path,
    defense_path: Path,
) -> pd.DataFrame:
    """Input: Thomas advanced/usage/defense CSV paths. Output: canonical player-season advanced feature table."""
    if not advanced_path.exists():
        return pd.DataFrame()

    advanced = pd.read_csv(advanced_path, low_memory=False)
    merge_keys = ["PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "AGE", "MIN", "SEASON"]
    source = advanced.copy()

    if usage_path.exists():
        usage = pd.read_csv(usage_path, low_memory=False)
        usage_keep = [column for column in [*merge_keys, "USG_PCT", "PCT_STL", "PCT_BLK"] if column in usage.columns]
        source = source.merge(usage[usage_keep], on=merge_keys, how="left", suffixes=("", "_USAGE"))

    if defense_path.exists():
        defense = pd.read_csv(defense_path, low_memory=False)
        defense_keep = [column for column in [*merge_keys, "DEF_RATING", "DREB_PCT", "STL", "PCT_STL", "BLK", "PCT_BLK"] if column in defense.columns]
        source = source.merge(defense[defense_keep], on=merge_keys, how="left", suffixes=("", "_DEFENSE"))

    def merged_column(name: str, fallback: object = np.nan) -> pd.Series:
        """Input: canonical source column name. Output: first non-null matching Series across merged suffixes."""
        candidates = [name, f"{name}_USAGE", f"{name}_DEFENSE"]
        existing = [column for column in candidates if column in source.columns]
        if not existing:
            return pd.Series(fallback, index=source.index)
        result = source[existing[0]]
        for column in existing[1:]:
            result = result.fillna(source[column])
        return result

    normalized = pd.DataFrame({
        "player_id": pd.to_numeric(source["PLAYER_ID"], errors="coerce").astype("Int64"),
        "player_name": source["PLAYER_NAME"],
        "season": source["SEASON"],
        "team_id": source["TEAM_ABBREVIATION"],
        "team_nba_id": source["TEAM_ID"],
        "team_abbreviation": source["TEAM_ABBREVIATION"],
        "age": pd.to_numeric(source["AGE"], errors="coerce"),
        "minutes": pd.to_numeric(source["MIN"], errors="coerce"),
        "usage_pct": pd.to_numeric(merged_column("USG_PCT"), errors="coerce"),
        "true_shooting_pct": pd.to_numeric(source.get("TS_PCT"), errors="coerce"),
        "turnover_rate": pd.to_numeric(source.get("TM_TOV_PCT"), errors="coerce"),
        "steal_rate": pd.to_numeric(merged_column("PCT_STL"), errors="coerce"),
        "block_rate": pd.to_numeric(merged_column("PCT_BLK"), errors="coerce"),
        "defensive_rebound_rate": pd.to_numeric(merged_column("DREB_PCT"), errors="coerce"),
        "pace": pd.to_numeric(source.get("PACE"), errors="coerce"),
        "possessions": pd.to_numeric(source.get("POSS"), errors="coerce"),
        "offensive_rating": pd.to_numeric(source.get("OFF_RATING"), errors="coerce"),
        "defensive_rating": pd.to_numeric(merged_column("DEF_RATING"), errors="coerce"),
    })
    return normalized.dropna(subset=["player_id"]).drop_duplicates(["player_id", "season", "team_id"]).reset_index(drop=True)

def player_id_lookup_from_players(players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical players table. Output: player name key to NBA player_id lookup."""
    if players_df.empty or "player_name" not in players_df.columns:
        return pd.DataFrame(columns=["player_name_key", "player_id"])
    lookup = players_df[["player_id", "player_name"]].dropna().drop_duplicates("player_id").copy()
    lookup["player_name_key"] = lookup["player_name"].map(normalize_name_key)
    return lookup[["player_name_key", "player_id"]].dropna().drop_duplicates("player_name_key")

def normalize_rodney_advanced_patch(path: Path, players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: Rodney Basketball-Reference advanced CSV and players. Output: 2023-24 advanced patch table."""
    if not path.exists():
        return pd.DataFrame()
    raw = pd.read_csv(path, low_memory=False)
    raw = raw[(raw["lg"].eq("NBA")) & (raw["season"].eq(2024)) & (~raw["tm"].eq("TOT"))].copy()
    raw["player_name_key"] = raw["player"].map(normalize_name_key)
    raw = raw.merge(player_id_lookup_from_players(players_df), on="player_name_key", how="left")
    normalized = pd.DataFrame({
        "player_id": raw["player_id_y"],
        "player_name": raw["player"],
        "season": raw["season"].map(season_label_from_bref_end_year),
        "team_id": raw["tm"].map(normalize_team_abbreviation),
        "age": pd.to_numeric(raw["age"], errors="coerce"),
        "minutes": pd.to_numeric(raw["mp"], errors="coerce"),
        "usage_pct": percent_to_ratio(raw["usg_percent"]),
        "true_shooting_pct": percent_to_ratio(raw["ts_percent"]),
        "three_point_attempt_rate": percent_to_ratio(raw["x3p_ar"]),
        "free_throw_rate": percent_to_ratio(raw["f_tr"]),
        "turnover_rate": percent_to_ratio(raw["tov_percent"]),
        "steal_rate": percent_to_ratio(raw["stl_percent"]),
        "block_rate": percent_to_ratio(raw["blk_percent"]),
        "defensive_rebound_rate": percent_to_ratio(raw["drb_percent"]),
        "offensive_rating": pd.NA,
        "defensive_rating": pd.NA,
    })
    return normalized.dropna(subset=["player_id"]).drop_duplicates(["player_id", "season", "team_id"]).reset_index(drop=True)

def normalize_ratin_2024_25_advanced_patch(
    advanced_path: Path,
    totals_path: Path,
    players_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: Ratin21 2024-25 advanced/totals CSVs and players. Output: 2024-25 advanced patch table."""
    if not advanced_path.exists():
        return pd.DataFrame()
    advanced = pd.read_csv(advanced_path, low_memory=False)
    if totals_path.exists():
        totals = pd.read_csv(totals_path, low_memory=False)
        raw = advanced.merge(totals[["Player", "Age", "Team", "Pos", "MP"]], on="Player", how="left")
    else:
        raw = advanced.copy()
        raw["Age"] = pd.NA
        raw["Team"] = pd.NA
        raw["MP"] = pd.NA

    raw["player_name_key"] = raw["Player"].map(normalize_name_key)
    raw = raw.merge(player_id_lookup_from_players(players_df), on="player_name_key", how="left")
    normalized = pd.DataFrame({
        "player_id": raw["player_id"],
        "player_name": raw["Player"],
        "season": "2024-25",
        "team_id": raw["Team"].map(normalize_team_abbreviation),
        "age": pd.to_numeric(raw["Age"], errors="coerce"),
        "minutes": pd.to_numeric(raw["MP"], errors="coerce"),
        "usage_pct": percent_to_ratio(raw["USG%"]),
        "true_shooting_pct": percent_to_ratio(raw["TS%"]),
        "offensive_rating": pd.NA,
        "defensive_rating": pd.NA,
    })
    return normalized.dropna(subset=["player_id"]).drop_duplicates(["player_id", "season", "team_id"]).reset_index(drop=True)

def build_advanced_patch_stats(players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: players table. Output: combined 2023-24 and 2024-25 advanced patch stats."""
    frames = [
        normalize_rodney_advanced_patch(RODNEY_ADVANCED_PATH, players_df),
        normalize_ratin_2024_25_advanced_patch(RATIN_ADVANCED_2024_25_PATH, RATIN_TOTAL_2024_25_PATH, players_df),
    ]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def enrich_season_stats_with_advanced(
    baseline_df: pd.DataFrame,
    advanced_df: pd.DataFrame,
    players_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: baseline box-score stats, advanced stats, players. Output: enriched player-season stats table."""
    if baseline_df.empty:
        return advanced_df
    if advanced_df.empty:
        return baseline_df

    enriched = baseline_df.merge(
        advanced_df,
        on=["player_id", "season", "team_id"],
        how="left",
        suffixes=("", "_advanced"),
    )

    fallback_cols = [
        "player_id", "season", "age", "minutes", "usage_pct", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate",
        "pace", "possessions", "offensive_rating", "defensive_rating",
    ]
    fallback_cols = [column for column in fallback_cols if column in advanced_df.columns]
    if {"player_id", "season"}.issubset(fallback_cols):
        fallback = (
            advanced_df[fallback_cols]
            .sort_values(["player_id", "season", "minutes" if "minutes" in advanced_df.columns else "player_id"], ascending=[True, True, False])
            .drop_duplicates(["player_id", "season"])
        )
        enriched = enriched.merge(fallback, on=["player_id", "season"], how="left", suffixes=("", "_fallback"))

    for column in [
        "player_name", "age", "minutes", "usage_pct", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate",
        "pace", "possessions", "offensive_rating", "defensive_rating",
    ]:
        advanced_column = f"{column}_advanced"
        fallback_column = f"{column}_fallback"
        if advanced_column in enriched.columns:
            enriched[column] = enriched[column].fillna(enriched[advanced_column])
        if fallback_column in enriched.columns:
            enriched[column] = enriched[column].fillna(enriched[fallback_column])

    if not players_df.empty and "birth_date" in players_df.columns:
        bio = players_df[["player_id", "birth_date"]].dropna().drop_duplicates("player_id")
        enriched = enriched.merge(bio, on="player_id", how="left")
        season_start = enriched["season"].astype(str).str.slice(0, 4).astype(float)
        birth_year = pd.to_datetime(enriched["birth_date"], errors="coerce").dt.year
        enriched["age"] = enriched["age"].fillna(season_start - birth_year)
        enriched = enriched.drop(columns=["birth_date"])

    keep_cols = [
        "player_id", "player_name", "season", "team_id", "age", "minutes", "usage_pct",
        "points_per_100", "assists_per_100", "rebounds_per_100", "true_shooting_pct",
        "three_point_attempt_rate", "free_throw_rate", "turnover_rate", "steal_rate", "block_rate",
        "defensive_rebound_rate", "foul_rate", "pace", "possessions", "offensive_rating", "defensive_rating",
    ]
    keep_cols = [column for column in keep_cols if column in enriched.columns]
    return enriched[keep_cols].sort_values(["season", "player_id", "team_id"]).reset_index(drop=True)

def needs_advanced_refresh(path: Path) -> bool:
    """Input: canonical season stats path. Output: True when advanced source should refresh sparse rate columns."""
    if should_fetch(path):
        return True
    current = read_table(path)
    if current.empty:
        return True
    target_patch_seasons = ["2023-24", "2024-25"]
    if "season" in current.columns and "usage_pct" in current.columns:
        target_rows = current[current["season"].isin(target_patch_seasons)]
        target_usage = pd.to_numeric(target_rows["usage_pct"], errors="coerce")
        if not target_rows.empty and target_usage.isna().mean() > 0.25:
            print("Refreshing season stats because advanced patch seasons are still sparse.")
            return True
        if target_usage.dropna().gt(1.5).any():
            print("Refreshing season stats because patched usage_pct is still on percentage scale.")
            return True
    key_cols = ["usage_pct", "defensive_rebound_rate", "offensive_rating", "defensive_rating"]
    existing = [column for column in key_cols if column in current.columns]
    if len(existing) < len(key_cols):
        return True
    return current[existing].isna().mean().mean() > 0.25

def season_start_year(season: str) -> int:
    """Input: season label such as 2024-25. Output: starting year as integer."""
    return int(str(season).split("-")[0])

def cache_safe_season(season: str) -> str:
    """Input: season label. Output: filesystem-safe token used in cache file names."""
    return str(season).replace("-", "_")

def nba_api_get_data_frame(endpoint_factory, label: str) -> pd.DataFrame:
    """Input: nba_api endpoint factory and label. Output: first DataFrame with timeout retries."""
    import time
    from requests.exceptions import ReadTimeout, Timeout

    last_error: Exception | None = None
    for attempt in range(1, NBA_API_MAX_RETRIES + 1):
        try:
            endpoint = endpoint_factory()
            return endpoint.get_data_frames()[0]
        except (ReadTimeout, Timeout, TimeoutError) as exc:
            last_error = exc
            wait_seconds = REQUEST_SLEEP_SECONDS * attempt
            print(f"Timeout while fetching {label}; retry {attempt}/{NBA_API_MAX_RETRIES} after {wait_seconds:.0f}s")
            time.sleep(wait_seconds)

    print(f"Failed to fetch {label}: {last_error}")
    return pd.DataFrame()

def normalize_nba_players() -> pd.DataFrame:
    """Input: none. Output: minimal players table from nba_api static player list."""
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    raw_players = pd.DataFrame(nba_static_players.get_players())
    players_df = raw_players.rename(
        columns={
            "id": "player_id",
            "full_name": "player_name",
        }
    )
    players_df["birth_date"] = pd.NaT
    players_df["position"] = pd.NA
    players_df["height"] = pd.NA
    players_df["weight"] = pd.NA
    return players_df[["player_id", "player_name", "birth_date", "position", "height", "weight"]]

def normalize_player_game_logs(logs: pd.DataFrame) -> pd.DataFrame:
    """Input: raw nba_api PlayerGameLogs DataFrame. Output: canonical player_game_logs DataFrame."""
    if logs.empty:
        return pd.DataFrame()

    normalized = pd.DataFrame({
        "player_id": logs["PLAYER_ID"],
        "player_name": logs.get("PLAYER_NAME"),
        "game_date": pd.to_datetime(logs["GAME_DATE"]),
        "game_id": logs["GAME_ID"],
        "season": logs["REQUESTED_SEASON"],
        "team_id": logs["TEAM_ID"],
        "team_abbreviation": logs.get("TEAM_ABBREVIATION"),
        "minutes": logs["MIN"],
        "points": logs["PTS"],
        "assists": logs["AST"],
        "rebounds": logs["REB"],
        "usage_pct": pd.NA,
        "true_shooting_pct": pd.NA,
        "opponent": logs["MATCHUP"].astype(str).str.extract(r"(?:vs\.|@)\s+([A-Z]{2,3})", expand=False),
        "home_away": np.where(logs["MATCHUP"].astype(str).str.contains(" @ ", regex=False), "AWAY", "HOME"),
    })
    normalized = normalized.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    normalized["rest_days"] = normalized.groupby("player_id")["game_date"].diff().dt.days
    return normalized

def fetch_league_player_game_logs(seasons: list[str]) -> pd.DataFrame:
    """Input: season labels. Output: combined nba_api game logs; fallback only, not primary source."""
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    frames = []
    for season in seasons:
        cache_path = NBA_API_CACHE_DIR / f"player_game_logs_{cache_safe_season(season)}.parquet"
        if cache_path.exists() and not OVERWRITE_EXISTING_NBA_API_RAW:
            print(f"Using cached player game logs: {season}")
            frames.append(pd.read_parquet(cache_path))
            continue

        print(f"Fetching player game logs: {season}")
        raw = nba_api_get_data_frame(
            lambda: playergamelogs.PlayerGameLogs(
                season_nullable=season,
                season_type_nullable=SEASON_TYPE,
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player game logs {season}",
        )
        if raw.empty:
            continue
        raw["REQUESTED_SEASON"] = season
        raw.to_parquet(cache_path, index=False)
        frames.append(raw)

    if not frames:
        return pd.DataFrame()

    return normalize_player_game_logs(pd.concat(frames, ignore_index=True))

def normalize_player_season_stats(stats: pd.DataFrame) -> pd.DataFrame:
    """Input: raw nba_api season stats. Output: canonical season_stats DataFrame."""
    if stats.empty:
        return pd.DataFrame()

    def col(name: str, default: float | None = np.nan) -> pd.Series:
        """Input: column name/default. Output: existing column or default-valued Series."""
        if name in stats.columns:
            return stats[name]
        return pd.Series(default, index=stats.index)

    fga = col("FGA").replace(0, np.nan)
    normalized = pd.DataFrame({
        "player_id": stats["PLAYER_ID"],
        "player_name": col("PLAYER_NAME", pd.NA),
        "season": stats["REQUESTED_SEASON"],
        "team_id": stats["TEAM_ID"],
        "age": col("AGE"),
        "minutes": col("MIN"),
        "usage_pct": col("USG_PCT"),
        "points_per_100": col("PTS"),
        "assists_per_100": col("AST"),
        "rebounds_per_100": col("REB"),
        "true_shooting_pct": col("TS_PCT"),
        "three_point_attempt_rate": col("FG3A") / fga,
        "free_throw_rate": col("FTA") / fga,
        "turnover_rate": col("TM_TOV_PCT").fillna(col("TOV_PCT")),
        "steal_rate": col("STL_PCT"),
        "block_rate": col("BLK_PCT"),
        "offensive_rating": col("OFF_RATING"),
        "defensive_rating": col("DEF_RATING"),
    })
    return normalized.sort_values(["season", "player_id"]).reset_index(drop=True)

def fetch_league_player_season_stats(seasons: list[str]) -> pd.DataFrame:
    """Input: season labels. Output: combined nba_api season stats; fallback only."""
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    frames = []
    for season in seasons:
        cache_path = NBA_API_CACHE_DIR / f"player_season_stats_{cache_safe_season(season)}.parquet"
        if cache_path.exists() and not OVERWRITE_EXISTING_NBA_API_RAW:
            print(f"Using cached player season stats: {season}")
            frames.append(pd.read_parquet(cache_path))
            continue

        print(f"Fetching player season stats: {season}")
        base = nba_api_get_data_frame(
            lambda: leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star=SEASON_TYPE,
                per_mode_detailed="Per100Possessions",
                measure_type_detailed_defense="Base",
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player season base stats {season}",
        )
        advanced = nba_api_get_data_frame(
            lambda: leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star=SEASON_TYPE,
                per_mode_detailed="Per100Possessions",
                measure_type_detailed_defense="Advanced",
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player season advanced stats {season}",
        )
        if base.empty or advanced.empty:
            continue
        merge_keys = [key for key in ["PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "AGE"] if key in base.columns and key in advanced.columns]
        raw = base.merge(advanced, on=merge_keys, how="left", suffixes=("", "_ADV"))
        raw["REQUESTED_SEASON"] = season
        raw.to_parquet(cache_path, index=False)
        frames.append(raw)

    if not frames:
        return pd.DataFrame()

    return normalize_player_season_stats(pd.concat(frames, ignore_index=True))


In [ ]:
def should_fetch(path: Path) -> bool:
    """Input: target path. Output: True when file is missing or overwrite flag is enabled."""
    return OVERWRITE_EXISTING_NBA_API_RAW or not path.exists()

def needs_game_logs_refresh(path: Path) -> bool:
    """Input: canonical game logs path. Output: True when cached logs miss required context/box-score columns."""
    if should_fetch(path):
        return True
    current_columns = set(pd.read_parquet(path, columns=None).columns)
    required_columns = {
        "opponent", "home_away", "steals", "blocks", "defensive_rebounds", "true_shooting_pct",
    }
    missing_columns = sorted(required_columns - current_columns)
    if missing_columns:
        print("Refreshing game logs because cached file is missing:", missing_columns)
        return True
    return False

if AUTO_FETCH_KAGGLE_TRADITIONAL_IF_MISSING and needs_game_logs_refresh(GAME_LOGS_PATH):
    data_path = NBA_TRADITIONAL_DIR
    if not any(data_path.rglob("*")):
        data_path = download_nba_traditional_dataset()

    player_boxscore_path = find_player_boxscore_file(data_path)
    print("Selected player box-score file:", player_boxscore_path)
    player_game_logs_raw = read_tabular_file(player_boxscore_path)
    fetched_game_logs = canonicalize_kaggle_player_game_logs(player_game_logs_raw)
    fetched_game_logs.to_parquet(GAME_LOGS_PATH, index=False)
    print(f"Saved {GAME_LOGS_PATH}: {fetched_game_logs.shape}")
elif GAME_LOGS_PATH.exists():
    print(f"Using existing {GAME_LOGS_PATH}")

if PLAYER_BIO_PATH.exists() and needs_player_bio_refresh(PLAYERS_PATH):
    players_from_bio = normalize_eoin_players(PLAYER_BIO_PATH)
    players_from_bio.to_parquet(PLAYERS_PATH, index=False)
    print(f"Saved {PLAYERS_PATH} from bio source: {players_from_bio.shape}")
elif should_fetch(PLAYERS_PATH) and GAME_LOGS_PATH.exists():
    players_from_logs = build_players_from_game_logs(pd.read_parquet(GAME_LOGS_PATH))
    if not players_from_logs.empty:
        players_from_logs.to_parquet(PLAYERS_PATH, index=False)
        print(f"Saved {PLAYERS_PATH}: {players_from_logs.shape}")

if AUTO_FETCH_ADVANCED_PATCH_IF_MISSING and not RODNEY_ADVANCED_PATH.exists():
    download_kaggle_dataset(KAGGLE_ADVANCED_2023_DATASET, ADVANCED_PATCH_2023_DIR)
if AUTO_FETCH_ADVANCED_PATCH_IF_MISSING and not RATIN_ADVANCED_2024_25_PATH.exists():
    download_kaggle_dataset(KAGGLE_ADVANCED_2024_DATASET, ADVANCED_PATCH_2024_DIR)

if needs_advanced_refresh(SEASON_STATS_PATH) and GAME_LOGS_PATH.exists():
    players_for_age = read_table(PLAYERS_PATH)
    season_stats_from_logs = build_season_stats_from_game_logs(pd.read_parquet(GAME_LOGS_PATH))
    advanced_frames = [
        normalize_thomas_player_season_stats(ADVANCED_STATS_PATH, USAGE_STATS_PATH, DEFENSE_STATS_PATH),
        build_advanced_patch_stats(players_for_age),
    ]
    advanced_frames = [frame for frame in advanced_frames if not frame.empty]
    advanced_season_stats = pd.concat(advanced_frames, ignore_index=True) if advanced_frames else pd.DataFrame()
    season_stats_enriched = enrich_season_stats_with_advanced(season_stats_from_logs, advanced_season_stats, players_for_age)
    if not season_stats_enriched.empty:
        season_stats_enriched.to_parquet(SEASON_STATS_PATH, index=False)
        print(f"Saved {SEASON_STATS_PATH}: {season_stats_enriched.shape}")

# Fallback only. Historical game logs should prefer KaggleHub snapshots over live stats.nba.com requests.
if AUTO_FETCH_NBA_API_IF_MISSING:
    if not NBA_API_AVAILABLE:
        raise ImportError("Install nba_api first by running `%pip install nba-api`, then restart/re-run the notebook.")

    if should_fetch(PLAYERS_PATH):
        fetched_players = normalize_nba_players()
        fetched_players.to_parquet(PLAYERS_PATH, index=False)
        print(f"Saved {PLAYERS_PATH}: {fetched_players.shape}")
    else:
        print(f"Using existing {PLAYERS_PATH}")

    if should_fetch(NBA_API_GAME_LOGS_PATH):
        fetched_game_logs = fetch_league_player_game_logs(FETCH_SEASONS)
        fetched_game_logs.to_parquet(NBA_API_GAME_LOGS_PATH, index=False)
        print(f"Saved {NBA_API_GAME_LOGS_PATH}: {fetched_game_logs.shape}")
    else:
        print(f"Using existing {NBA_API_GAME_LOGS_PATH}")

    if should_fetch(SEASON_STATS_PATH):
        fetched_season_stats = fetch_league_player_season_stats(FETCH_SEASONS)
        fetched_season_stats.to_parquet(SEASON_STATS_PATH, index=False)
        print(f"Saved {SEASON_STATS_PATH}: {fetched_season_stats.shape}")
    else:
        print(f"Using existing {SEASON_STATS_PATH}")

missing_salary_sources = [path for path in [PLAYER_SEASON_SALARIES_PATH, SALARY_CAP_PATH] if not path.exists()]
if missing_salary_sources:
    print("Additional non-nba_api data status for salary analysis:")
    for path in missing_salary_sources:
        print(f"  {path}")


In [ ]:
players = read_table(PLAYERS_PATH)
game_logs = read_table(GAME_LOGS_PATH)
season_stats = read_table(SEASON_STATS_PATH)
salary_cap = read_table(SALARY_CAP_PATH)
player_season_salaries = read_table(PLAYER_SEASON_SALARIES_PATH)

def build_default_salary_cap_by_season() -> pd.DataFrame:
    """Input: none. Output: curated NBA salary-cap table used when Drive raw cap data is incomplete."""
    rows = [
        ("1999-00", 34000000, None),
        ("2000-01", 35500000, None),
        ("2001-02", 42500000, None),
        ("2002-03", 40271000, 52880000),
        ("2003-04", 43840000, 54556722),
        ("2004-05", 43870000, None),
        ("2005-06", 49500000, 61700000),
        ("2006-07", 53135000, 65420000),
        ("2007-08", 55630000, 67865000),
        ("2008-09", 58680000, 71150000),
        ("2009-10", 57700000, 69920000),
        ("2010-11", 58044000, 70307000),
        ("2011-12", 58044000, 70307000),
        ("2012-13", 58044000, 70307000),
        ("2013-14", 58679000, 71748000),
        ("2014-15", 63065000, 76829000),
        ("2015-16", 70000000, 84740000),
        ("2016-17", 94143000, 113287000),
        ("2017-18", 99093000, 119266000),
        ("2018-19", 101869000, 123733000),
        ("2019-20", 109140000, 132627000),
        ("2020-21", 109140000, 132627000),
        ("2021-22", 112414000, 136606000),
        ("2022-23", 123655000, 150267000),
        ("2023-24", 136021000, 165294000),
        ("2024-25", 140588000, 170814000),
        ("2025-26", 154647000, 187895000),
    ]
    return pd.DataFrame(rows, columns=["season", "salary_cap_usd", "tax_level_usd"]).assign(source="curated_salary_cap_history")

def complete_salary_cap_table(salary_cap_df: pd.DataFrame) -> pd.DataFrame:
    """Input: raw salary-cap table. Output: complete canonical salary-cap table for all salary seasons."""
    default_cap = build_default_salary_cap_by_season()
    if salary_cap_df.empty:
        cap = default_cap
    else:
        cap = salary_cap_df.copy()
        if "salary_cap" in cap.columns and "salary_cap_usd" not in cap.columns:
            cap = cap.rename(columns={"salary_cap": "salary_cap_usd"})
        for column in ["tax_level_usd", "source"]:
            if column not in cap.columns:
                cap[column] = pd.NA
        cap = pd.concat([default_cap, cap[["season", "salary_cap_usd", "tax_level_usd", "source"]]], ignore_index=True)
        cap = cap.drop_duplicates("season", keep="last")
    cap["season_start_year"] = cap["season"].astype(str).str.slice(0, 4).astype(int)
    return cap.sort_values("season_start_year").drop(columns=["season_start_year"]).reset_index(drop=True)

salary_cap = complete_salary_cap_table(salary_cap)

tables = {
    "players": players,
    "player_game_logs": game_logs,
    "player_season_stats": season_stats,
    "salary_cap_by_season": salary_cap,
    "player_season_salaries": player_season_salaries,
}

for table_name, table_df in tables.items():
    report_schema(table_name, table_df, RAW_SCHEMAS[table_name])


## 3. Gold Dataset 1: Player Role Features

Each row represents one player in one season. The table supports role profiling, similarity search, candidate retrieval, explanation features, and salary modeling joins.

In [ ]:
ROLE_BASE_FEATURES = [
    "minutes",
    "usage_pct",
    "points_per_100",
    "assists_per_100",
    "rebounds_per_100",
    "true_shooting_pct",
    "three_point_attempt_rate",
    "free_throw_rate",
    "turnover_rate",
    "steal_rate",
    "block_rate",
    "defensive_rebound_rate",
    "foul_rate",
    "pace",
    "possessions",
    "offensive_rating",
    "defensive_rating",
]

def add_role_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    """Input: season stats with base role features. Output: DataFrame with derived role dimensions."""
    role = df.copy()
    role["scoring_creation"] = role[["points_per_100", "usage_pct", "free_throw_rate"]].mean(axis=1)
    role["playmaking"] = role[["assists_per_100", "usage_pct"]].mean(axis=1) - role["turnover_rate"].fillna(0)
    role["shooting"] = role[["true_shooting_pct", "three_point_attempt_rate"]].mean(axis=1)
    role["rim_pressure"] = role[["free_throw_rate", "points_per_100"]].mean(axis=1)
    role["rebounding"] = role[["rebounds_per_100", "defensive_rebound_rate"]].mean(axis=1)
    role["perimeter_defense"] = role["steal_rate"]
    role["interior_defense"] = role[["block_rate", "defensive_rebound_rate"]].mean(axis=1)
    role["two_way_impact"] = role["offensive_rating"] - role["defensive_rating"]
    return role

ROLE_DIMENSIONS = [
    "scoring_creation", "playmaking", "shooting", "rim_pressure", "rebounding",
    "perimeter_defense", "interior_defense", "two_way_impact",
]

def build_player_role_features(season_stats_df: pd.DataFrame, players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: season_stats and players. Output: player-season role feature table for similarity."""
    if season_stats_df.empty:
        return pd.DataFrame()

    required = ["player_id", "season", "team_id", "age"] + ROLE_BASE_FEATURES
    role = season_stats_df[required].copy()
    role = add_role_dimensions(role)

    identity_cols = [col for col in ["player_id", "player_name", "position"] if col in players_df.columns]
    if identity_cols:
        role = role.merge(players_df[identity_cols].drop_duplicates("player_id"), on="player_id", how="left")

    output_cols = [
        "player_id", "player_name", "season", "team_id", "age", "position",
        *ROLE_BASE_FEATURES,
        *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in role.columns]
    return role[output_cols].sort_values(["season", "player_id"]).reset_index(drop=True)

player_role_features = build_player_role_features(season_stats, players)
player_role_features.head()


In [ ]:
if not player_role_features.empty:
    player_role_features.to_parquet(OUTPUT_ROLE_FEATURES, index=False)
    print(f"Saved {OUTPUT_ROLE_FEATURES} with shape {player_role_features.shape}")


### Quick Similarity Prototype

This prototype checks player-role similarity for a target player and season. Adjust `TARGET_PLAYER_NAME`, `TARGET_SEASON`, and `SIMILARITY_WEIGHTS` to inspect candidate matches.

In [ ]:
TARGET_PLAYER_NAME = "LeBron James"
TARGET_SEASON = None

SIMILARITY_WEIGHTS = {
    "scoring_creation": 1.0,
    "playmaking": 1.2,
    "shooting": 0.8,
    "rim_pressure": 1.0,
    "rebounding": 0.7,
    "perimeter_defense": 0.8,
    "interior_defense": 0.4,
    "two_way_impact": 0.8,
}

def find_similar_players(
    role_df: pd.DataFrame,
    target_player_name: str,
    target_season: str | int | None = None,
    top_k: int = 10,
    weights: dict[str, float] | None = None,
) -> pd.DataFrame:
    """Input: role table, target player/season, top_k, weights. Output: top similar player-seasons."""
    if role_df.empty:
        return pd.DataFrame()

    features = [feature for feature in ROLE_DIMENSIONS if feature in role_df.columns]
    matrix = role_df[features].copy()
    if weights:
        for feature, weight in weights.items():
            if feature in matrix.columns:
                matrix[feature] = matrix[feature] * weight

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X = preprocessor.fit_transform(matrix)

    target_mask = role_df["player_name"].str.lower().eq(target_player_name.lower())
    if target_season is not None:
        target_mask &= role_df["season"].eq(target_season)
    if not target_mask.any():
        raise ValueError(f"Target player not found: {target_player_name}, season={target_season}")

    target_idx = role_df[target_mask].index[-1]
    sims = cosine_similarity(X[target_idx : target_idx + 1], X).ravel()

    results = role_df.copy()
    results["similarity_score"] = sims
    results = results.loc[results.index != target_idx]
    return results.sort_values("similarity_score", ascending=False).head(top_k)

if not player_role_features.empty and "player_name" in player_role_features.columns:
    similar_players = find_similar_players(
        player_role_features,
        TARGET_PLAYER_NAME,
        target_season=TARGET_SEASON,
        top_k=10,
        weights=SIMILARITY_WEIGHTS,
    )
    display(similar_players[["player_name", "season", "team_id", "position", "similarity_score", *ROLE_DIMENSIONS]])


## 4. Gold Dataset 2: Performance Training

Each row represents one player at one `as_of_date`. Features use only information available through that date, while targets measure average production across the next five games.

In [ ]:
def add_future_average(
    df: pd.DataFrame,
    group_cols: list[str],
    column: str,
    output_name: str,
    horizon: int = 5,
) -> pd.DataFrame:
    """Input: ordered table, group columns, raw stat column, output name, horizon. Output: table with target t+1..t+h average."""
    result = df.sort_values([*group_cols, "game_date", "game_id"]).copy()
    grouped_values = result.groupby(group_cols, sort=False)[column]
    future_values = [grouped_values.shift(-step) for step in range(1, horizon + 1)]
    result[output_name] = sum(future_values) / horizon
    return result

def add_rolling_player_features(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical game logs. Output: point-in-time tabular features, raw sequence values, and next-5-average targets."""
    if game_logs_df.empty:
        return pd.DataFrame()

    df = game_logs_df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"])
    df = df.sort_values(["player_id", "season", "game_date", "game_id"]).reset_index(drop=True)

    df["pts"] = pd.to_numeric(df["points"], errors="coerce")
    df["ast"] = pd.to_numeric(df["assists"], errors="coerce")
    df["reb"] = pd.to_numeric(df["rebounds"], errors="coerce")
    df["min"] = pd.to_numeric(df["minutes"], errors="coerce")

    player_season_grouped = df.groupby(["player_id", "season"], group_keys=False)
    stat_prefixes = {"pts": "pts", "ast": "ast", "reb": "reb"}

    for stat, prefix in stat_prefixes.items():
        for window in [5, 10]:
            df[f"{prefix}_last_{window}"] = player_season_grouped[stat].transform(
                lambda s, window=window: s.rolling(window=window, min_periods=window).mean()
            )
        df[f"{prefix}_season_avg"] = player_season_grouped[stat].transform(
            lambda s: s.expanding(min_periods=1).mean()
        )

    for window in [5, 10]:
        df[f"min_last_{window}"] = player_season_grouped["min"].transform(
            lambda s, window=window: s.rolling(window=window, min_periods=window).mean()
        )
    df["min_season_avg"] = player_season_grouped["min"].transform(
        lambda s: s.expanding(min_periods=1).mean()
    )

    for prefix in ["pts", "ast", "reb"]:
        df[f"{prefix}_last_5_minus_season_avg"] = df[f"{prefix}_last_5"] - df[f"{prefix}_season_avg"]
        df[f"{prefix}_last_10_minus_season_avg"] = df[f"{prefix}_last_10"] - df[f"{prefix}_season_avg"]
        df[f"{prefix}_last_5_minus_last_10"] = df[f"{prefix}_last_5"] - df[f"{prefix}_last_10"]
    df["min_last_5_minus_season_avg"] = df["min_last_5"] - df["min_season_avg"]
    df["min_last_10_minus_season_avg"] = df["min_last_10"] - df["min_season_avg"]
    df["min_last_5_minus_last_10"] = df["min_last_5"] - df["min_last_10"]

    target_specs = {
        "pts": "target_next_5_pts_avg",
        "ast": "target_next_5_ast_avg",
        "reb": "target_next_5_reb_avg",
    }
    for raw_col, target_col in target_specs.items():
        df = add_future_average(
            df=df,
            group_cols=["player_id", "season"],
            column=raw_col,
            output_name=target_col,
            horizon=5,
        )

    df = df.rename(columns={"game_date": "as_of_date"})
    return df.reset_index(drop=True)

def build_performance_training(
    game_logs_df: pd.DataFrame,
    season_stats_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Input: canonical game logs. Output: leakage-safe performance_training table for tabular and sequence experiments."""
    # Do not merge full-season season_stats here; those aggregates include games after as_of_date.
    features = add_rolling_player_features(game_logs_df)
    if features.empty:
        return pd.DataFrame()

    id_cols = ["player_id", "as_of_date", "game_id", "season", "team_id"]
    raw_sequence_cols = ["pts", "ast", "reb", "min"]
    tabular_feature_cols = [
        "pts_last_5", "pts_last_10", "pts_season_avg",
        "pts_last_5_minus_season_avg", "pts_last_10_minus_season_avg", "pts_last_5_minus_last_10",
        "ast_last_5", "ast_last_10", "ast_season_avg",
        "ast_last_5_minus_season_avg", "ast_last_10_minus_season_avg", "ast_last_5_minus_last_10",
        "reb_last_5", "reb_last_10", "reb_season_avg",
        "reb_last_5_minus_season_avg", "reb_last_10_minus_season_avg", "reb_last_5_minus_last_10",
        "min_last_5", "min_last_10", "min_season_avg",
        "min_last_5_minus_season_avg", "min_last_10_minus_season_avg", "min_last_5_minus_last_10",
    ]
    target_cols = ["target_next_5_pts_avg", "target_next_5_ast_avg", "target_next_5_reb_avg"]
    output_cols = [col for col in [*id_cols, *raw_sequence_cols, *tabular_feature_cols, *target_cols] if col in features.columns]
    return features[output_cols].reset_index(drop=True)

performance_training = build_performance_training(game_logs)
performance_training.head()


In [ ]:
if not performance_training.empty:
    performance_training.to_parquet(OUTPUT_PERFORMANCE_TRAINING, index=False)
    print(f"Saved {OUTPUT_PERFORMANCE_TRAINING} with shape {performance_training.shape}")


## 5. Gold Dataset 3: Salary Analysis / Training

Each row represents one player in one salary season. The table focuses on `salary_usd` and `salary_cap_share` analysis, with role, production, and bio context joined at the season level.

When `salary_cap_by_season.csv` is available, the notebook adds:

```text
salary_cap_share = salary_usd / salary_cap_usd
```

Salary modeling should prioritize `salary_cap_share` as the target, then convert predictions back to USD using the salary cap for the analysis season.

In [ ]:
def normalize_name_for_join(value: object) -> str | None:
    """Input: player name. Output: normalized name key for fuzzy-light salary/role joins."""
    if pd.isna(value):
        return None
    return str(value).strip().lower().replace(".", "").replace("'", "")

def build_salary_training(
    player_salaries_df: pd.DataFrame,
    salary_cap_df: pd.DataFrame,
    role_features_df: pd.DataFrame,
    players_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: salaries, salary cap table, role features, players. Output: salary_training table."""
    if player_salaries_df.empty:
        return pd.DataFrame()

    salary = player_salaries_df.copy()
    salary["player_name_join"] = salary["player_name"].map(normalize_name_for_join)
    salary["target_salary_usd"] = salary["salary_usd"]

    if not salary_cap_df.empty:
        cap = salary_cap_df.copy()
        if "season" in cap.columns and "season_label" not in cap.columns:
            cap = cap.rename(columns={"season": "season_label"})
        if "salary_cap" in cap.columns and "salary_cap_usd" not in cap.columns:
            cap = cap.rename(columns={"salary_cap": "salary_cap_usd"})
        salary = salary.merge(cap[["season_label", "salary_cap_usd"]], on="season_label", how="left")
        salary["salary_cap_share"] = salary["salary_usd"] / salary["salary_cap_usd"]
    else:
        salary["salary_cap_usd"] = pd.NA
        salary["salary_cap_share"] = pd.NA

    if not role_features_df.empty and "player_name" in role_features_df.columns:
        role = role_features_df.copy()
        role["player_name_join"] = role["player_name"].map(normalize_name_for_join)
        role = role.rename(columns={"season": "season_label"})
        keep_cols = [
            "player_name_join", "season_label", "player_id", "team_id", "age", "position",
            *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS,
        ]
        keep_cols = [col for col in keep_cols if col in role.columns]
        salary = salary.merge(
            role[keep_cols].drop_duplicates(["player_name_join", "season_label"]),
            on=["player_name_join", "season_label"],
            how="left",
        )

    if not players_df.empty and "player_name" in players_df.columns:
        bio = players_df.copy()
        bio["player_name_join"] = bio["player_name"].map(normalize_name_for_join)
        bio_cols = ["player_name_join", "player_id", "birth_date", "position", "height", "weight"]
        bio_cols = [column for column in bio_cols if column in bio.columns]
        salary = salary.merge(
            bio[bio_cols].drop_duplicates("player_name_join"),
            on="player_name_join",
            how="left",
            suffixes=("", "_bio"),
        )
        if "player_id_bio" in salary.columns:
            salary["player_id"] = salary.get("player_id", pd.Series(pd.NA, index=salary.index)).fillna(salary["player_id_bio"])
        if "position_bio" in salary.columns:
            salary["position"] = salary.get("position", pd.Series(pd.NA, index=salary.index)).fillna(salary["position_bio"])
        if "age" not in salary.columns:
            salary["age"] = pd.NA
        if "birth_date" in salary.columns:
            birth_year = pd.to_datetime(salary["birth_date"], errors="coerce").dt.year
            salary["age"] = salary["age"].fillna(salary["season_start_year"] - birth_year)

    output_cols = [
        "player_id", "player_name", "team", "team_id", "season_start_year", "season_end_year",
        "season_label", "age", "position", "height", "weight", "salary_usd", "salary_cap_usd", "salary_cap_share",
        "target_salary_usd", "source", "source_file", "collected_at",
        *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in salary.columns]
    return salary[output_cols].sort_values(["season_start_year", "salary_usd", "player_name"], ascending=[True, False, True]).reset_index(drop=True)

salary_training = build_salary_training(player_season_salaries, salary_cap, player_role_features, players)
salary_training.head()


In [ ]:
if not salary_training.empty:
    salary_training.to_parquet(OUTPUT_SALARY_TRAINING, index=False)
    print(f"Saved {OUTPUT_SALARY_TRAINING} with shape {salary_training.shape}")


## 6. Clean Final DataFrames

This section creates the final clean dataframes used by downstream feature selection and modeling. Cleaning covers type normalization, percentage scaling, duplicate keys, invalid values, missing-value flags, and temporal split assignment.

In [ ]:
def normalize_percent_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Input: DataFrame and percent-like columns. Output: copy with mixed percent/ratio scales normalized to ratio."""
    cleaned = df.copy()
    for column in columns:
        if column in cleaned.columns:
            cleaned[column] = percent_to_ratio(cleaned[column])
    return cleaned

def coerce_numeric_columns(df: pd.DataFrame, exclude: list[str] | None = None) -> pd.DataFrame:
    """Input: DataFrame and excluded columns. Output: copy with likely numeric columns coerced to numeric dtype."""
    excluded = set(exclude or [])
    cleaned = df.copy()
    for column in cleaned.columns:
        if column in excluded:
            continue
        if pd.api.types.is_numeric_dtype(cleaned[column]):
            cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")
    return cleaned

def standardize_position(value: object) -> str:
    """Input: raw position value. Output: compact uppercase position string or UNK."""
    if pd.isna(value) or str(value).strip() == "":
        return "UNK"
    text = str(value).strip().upper().replace("-", "/")
    valid_parts = [part for part in text.split("/") if part in {"G", "F", "C", "PG", "SG", "SF", "PF"}]
    return "/".join(valid_parts) if valid_parts else "UNK"

def add_missing_flags_and_impute_numeric(
    df: pd.DataFrame,
    columns: list[str],
    group_col: str | None = None,
) -> pd.DataFrame:
    """Input: DataFrame, numeric columns, optional group. Output: copy with missing flags and median-imputed values."""
    cleaned = df.copy()
    for column in columns:
        if column not in cleaned.columns:
            continue
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")
        cleaned[f"{column}_was_missing"] = cleaned[column].isna()
        if group_col and group_col in cleaned.columns:
            grouped_median = cleaned.groupby(group_col)[column].transform("median")
            cleaned[column] = cleaned[column].fillna(grouped_median)
        global_median = cleaned[column].median()
        cleaned[column] = cleaned[column].fillna(0 if pd.isna(global_median) else global_median)
    return cleaned

def clip_existing_columns(df: pd.DataFrame, bounds: dict[str, tuple[float | None, float | None]]) -> pd.DataFrame:
    """Input: DataFrame and column bounds. Output: copy with existing numeric columns clipped to plausible ranges."""
    cleaned = df.copy()
    for column, (lower, upper) in bounds.items():
        if column in cleaned.columns:
            cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce").clip(lower=lower, upper=upper)
    return cleaned

def clean_player_role_features(df: pd.DataFrame) -> pd.DataFrame:
    """Input: player_role_features. Output: cleaned player-season-team role dataframe."""
    if df.empty:
        return pd.DataFrame()
    cleaned = df.copy()
    cleaned = cleaned.dropna(subset=["player_id", "season", "team_id"]).drop_duplicates(["player_id", "season", "team_id"])
    cleaned["player_id"] = pd.to_numeric(cleaned["player_id"], errors="coerce").astype("Int64")
    cleaned["season"] = cleaned["season"].astype(str)
    cleaned["team_id"] = cleaned["team_id"].astype(str).str.upper().str.strip()
    if "player_name" in cleaned.columns:
        cleaned["player_name"] = cleaned["player_name"].astype(str).str.strip()
    if "position" in cleaned.columns:
        cleaned["position"] = cleaned["position"].map(standardize_position)
    cleaned = normalize_percent_columns(cleaned, [
        "usage_pct", "true_shooting_pct", "three_point_attempt_rate", "free_throw_rate", "turnover_rate",
    ])
    cleaned = clip_existing_columns(cleaned, {
        "age": (15, 50), "minutes": (0, None), "usage_pct": (0, 0.6), "true_shooting_pct": (0, 1.5),
        "three_point_attempt_rate": (0, 1), "free_throw_rate": (0, 2), "turnover_rate": (0, 1),
        "points_per_100": (0, 100), "assists_per_100": (0, 60), "rebounds_per_100": (0, 60),
        "steal_rate": (0, 10), "block_rate": (0, 10), "defensive_rebound_rate": (0, 40),
        "foul_rate": (0, 15), "pace": (80, 120), "possessions": (0, None),
        "offensive_rating": (50, 160), "defensive_rating": (50, 160),
    })
    numeric_cols = cleaned.select_dtypes(include="number").columns.difference(["player_id"]).tolist()
    cleaned = add_missing_flags_and_impute_numeric(cleaned, numeric_cols, group_col="season")
    return cleaned.sort_values(["season", "player_id", "team_id"]).reset_index(drop=True)

def clean_performance_training(df: pd.DataFrame) -> pd.DataFrame:
    """Input: performance_training. Output: complete-case dataframe for tabular and sequence forecasting experiments."""
    if df.empty:
        return pd.DataFrame()
    cleaned = df.copy()
    cleaned = cleaned.dropna(subset=["player_id", "as_of_date", "game_id", "season", "team_id"])
    cleaned = cleaned.drop_duplicates(["player_id", "as_of_date", "game_id", "team_id"])
    cleaned["player_id"] = pd.to_numeric(cleaned["player_id"], errors="coerce").astype("Int64")
    cleaned["as_of_date"] = pd.to_datetime(cleaned["as_of_date"], errors="coerce")
    cleaned["season"] = cleaned["season"].astype(str)
    cleaned["team_id"] = cleaned["team_id"].astype(str).str.upper().str.strip()

    raw_sequence_cols = ["pts", "ast", "reb", "min"]
    tabular_feature_cols = [
        "pts_last_5", "pts_last_10", "pts_season_avg",
        "pts_last_5_minus_season_avg", "pts_last_10_minus_season_avg", "pts_last_5_minus_last_10",
        "ast_last_5", "ast_last_10", "ast_season_avg",
        "ast_last_5_minus_season_avg", "ast_last_10_minus_season_avg", "ast_last_5_minus_last_10",
        "reb_last_5", "reb_last_10", "reb_season_avg",
        "reb_last_5_minus_season_avg", "reb_last_10_minus_season_avg", "reb_last_5_minus_last_10",
        "min_last_5", "min_last_10", "min_season_avg",
        "min_last_5_minus_season_avg", "min_last_10_minus_season_avg", "min_last_5_minus_last_10",
    ]
    target_cols = [column for column in cleaned.columns if column.startswith("target_next_5_") and column.endswith("_avg")]
    required_cols = [column for column in [*raw_sequence_cols, *tabular_feature_cols, *target_cols] if column in cleaned.columns]
    before = len(cleaned)
    cleaned = cleaned.dropna(subset=required_cols, how="any").copy()
    print(f"Dropped incomplete performance samples: {before:,} -> {len(cleaned):,}")
    cleaned = clip_existing_columns(cleaned, {
        "pts": (0, 100), "ast": (0, 40), "reb": (0, 40), "min": (0, 60),
        "min_last_5": (0, 48), "min_last_10": (0, 48), "min_season_avg": (0, 48),
        "min_last_5_minus_season_avg": (-48, 48), "min_last_10_minus_season_avg": (-48, 48),
        "min_last_5_minus_last_10": (-48, 48),
    })
    keep_cols = [
        "player_id", "as_of_date", "game_id", "season", "team_id",
        *raw_sequence_cols, *tabular_feature_cols, *target_cols,
    ]
    keep_cols = [column for column in keep_cols if column in cleaned.columns]
    return cleaned[keep_cols].sort_values(["season", "player_id", "as_of_date", "game_id"]).reset_index(drop=True)

def clean_salary_training(df: pd.DataFrame) -> pd.DataFrame:
    """Input: salary_training. Output: cleaned player-season salary dataframe."""
    if df.empty:
        return pd.DataFrame()
    cleaned = df.copy()
    cleaned = cleaned.dropna(subset=["player_name", "season_label", "salary_usd", "salary_cap_usd", "salary_cap_share"])
    cleaned = cleaned.drop_duplicates(["player_name", "season_label", "team", "salary_usd"])
    if "player_id" in cleaned.columns:
        cleaned["player_id_was_missing"] = cleaned["player_id"].isna()
        cleaned["player_id"] = pd.to_numeric(cleaned["player_id"], errors="coerce").fillna(-1).astype("Int64")
    cleaned["player_name"] = cleaned["player_name"].astype(str).str.strip()
    cleaned["season_label"] = cleaned["season_label"].astype(str)
    for column in ["team", "team_id"]:
        if column in cleaned.columns:
            cleaned[column] = cleaned[column].fillna("UNK").astype(str).str.upper().str.strip()
    if "position" in cleaned.columns:
        cleaned["position"] = cleaned["position"].map(standardize_position)
    for column in ["source", "source_file", "collected_at"]:
        if column in cleaned.columns:
            cleaned[column] = cleaned[column].fillna("unknown")
    cleaned = normalize_percent_columns(cleaned, [
        "usage_pct", "true_shooting_pct", "three_point_attempt_rate", "free_throw_rate", "turnover_rate", "salary_cap_share",
    ])
    cleaned = cleaned[
        (pd.to_numeric(cleaned["salary_usd"], errors="coerce") > 0)
        & (pd.to_numeric(cleaned["salary_cap_usd"], errors="coerce") > 0)
        & (pd.to_numeric(cleaned["salary_cap_share"], errors="coerce").between(0, 1.5))
    ].copy()
    cleaned = clip_existing_columns(cleaned, {
        "age": (15, 50), "height": (60, 96), "weight": (130, 360), "salary_usd": (1, None),
        "salary_cap_usd": (1, None), "salary_cap_share": (0, 1.5), "usage_pct": (0, 0.6),
        "true_shooting_pct": (0, 1.5), "three_point_attempt_rate": (0, 1), "free_throw_rate": (0, 2),
        "turnover_rate": (0, 1), "steal_rate": (0, 10), "block_rate": (0, 10),
        "defensive_rebound_rate": (0, 40), "foul_rate": (0, 15), "pace": (80, 120),
        "offensive_rating": (50, 160), "defensive_rating": (50, 160),
    })
    numeric_cols = cleaned.select_dtypes(include="number").columns.difference(["player_id"]).tolist()
    cleaned = add_missing_flags_and_impute_numeric(cleaned, numeric_cols, group_col="season_label")
    return cleaned.sort_values(["season_start_year", "salary_usd", "player_name"], ascending=[True, False, True]).reset_index(drop=True)

def assert_clean_dataframe(name: str, df: pd.DataFrame, key_cols: list[str]) -> None:
    """Input: clean DataFrame name, DataFrame, key columns. Output: assertion errors or printed clean summary."""
    if df.empty:
        raise ValueError(f"{name} is empty after cleaning.")
    missing_keys = [column for column in key_cols if column not in df.columns]
    if missing_keys:
        raise ValueError(f"{name} is missing key columns: {missing_keys}")
    if df[key_cols].isna().any().any():
        raise ValueError(f"{name} has missing key values.")
    duplicate_count = df.duplicated(key_cols).sum()
    if duplicate_count:
        raise ValueError(f"{name} has {duplicate_count:,} duplicate key rows.")
    print(f"{name}: clean shape {df.shape}")

player_role_features_clean = clean_player_role_features(player_role_features)
performance_training_clean = clean_performance_training(performance_training)
salary_training_clean = clean_salary_training(salary_training)

assert_clean_dataframe("player_role_features_clean", player_role_features_clean, ["player_id", "season", "team_id"])
assert_clean_dataframe("performance_training_clean", performance_training_clean, ["player_id", "as_of_date", "game_id", "team_id"])
assert_clean_dataframe("salary_training_clean", salary_training_clean, ["player_name", "season_label", "team", "salary_usd"])

player_role_features_clean.head(), performance_training_clean.head(), salary_training_clean.head()


In [ ]:
if not player_role_features_clean.empty:
    player_role_features_clean.to_parquet(OUTPUT_ROLE_FEATURES_CLEAN, index=False)
    print(f"Saved {OUTPUT_ROLE_FEATURES_CLEAN} with shape {player_role_features_clean.shape}")

if not performance_training_clean.empty:
    performance_training_clean.to_parquet(OUTPUT_PERFORMANCE_TRAINING_CLEAN, index=False)
    print(f"Saved {OUTPUT_PERFORMANCE_TRAINING_CLEAN} with shape {performance_training_clean.shape}")

if not salary_training_clean.empty:
    salary_training_clean.to_parquet(OUTPUT_SALARY_TRAINING_CLEAN, index=False)
    print(f"Saved {OUTPUT_SALARY_TRAINING_CLEAN} with shape {salary_training_clean.shape}")


## 7. Basic Data Quality Checks

These checks summarize schema, missingness, unique values, and split coverage for the cleaned gold dataframes.

In [ ]:
def quality_summary(name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Input: table name and DataFrame. Output: column-level dtype/missing/unique summary."""
    if df.empty:
        print(f"{name}: empty")
        return pd.DataFrame()
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean(),
        "n_unique": df.nunique(dropna=True),
    })
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    return summary.sort_values("missing_pct", ascending=False)

display(quality_summary("player_role_features_clean", player_role_features_clean).head(20))
display(quality_summary("performance_training_clean", performance_training_clean).head(20))
display(quality_summary("salary_training_clean", salary_training_clean).head(20))


In [ ]:
def assign_temporal_split(season_label: object) -> str:
    """Input: season label. Output: train/validation/test split label for short-term and salary experiments."""
    if pd.isna(season_label):
        return "unknown"
    season = str(season_label)
    if season in TEST_SEASONS:
        return "test"
    if season in VALIDATION_SEASONS:
        return "validation"
    if season <= TRAIN_END_SEASON:
        return "train"
    return "future_or_unassigned"


def assign_long_term_temporal_split(season_label: object) -> str:
    """Input: anchor season label. Output: train/validation/test split with enough future seasons for h3 labels."""
    if pd.isna(season_label):
        return "unknown"
    season = str(season_label)
    if season in LONG_TERM_TEST_SEASONS:
        return "test"
    if season in LONG_TERM_VALIDATION_SEASONS:
        return "validation"
    if season <= LONG_TERM_TRAIN_END_SEASON:
        return "train"
    return "future_or_unassigned"

if not player_role_features_clean.empty and "season" in player_role_features_clean.columns:
    player_role_features_clean = player_role_features_clean.copy()
    player_role_features_clean["split"] = player_role_features_clean["season"].map(assign_temporal_split)
    player_role_features_clean.to_parquet(OUTPUT_ROLE_FEATURES_CLEAN, index=False)
    display(player_role_features_clean["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))

if not salary_training_clean.empty and "season_label" in salary_training_clean.columns:
    salary_training_clean = salary_training_clean.copy()
    salary_training_clean["split"] = salary_training_clean["season_label"].map(assign_temporal_split)
    salary_training_clean.to_parquet(OUTPUT_SALARY_TRAINING_CLEAN, index=False)
    display(salary_training_clean["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))

if not performance_training_clean.empty and "season" in performance_training_clean.columns:
    performance_training_clean = performance_training_clean.copy()
    performance_training_clean["split"] = performance_training_clean["season"].map(assign_temporal_split)
    performance_training_clean.to_parquet(OUTPUT_PERFORMANCE_TRAINING_CLEAN, index=False)
    display(performance_training_clean["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))


## 8. Performance Forecast Models

This section trains separate short-term models for points, assists, and rebounds. Metadata columns are retained for splitting and audit, but excluded from model feature matrices.

Experiment design:

- Tabular models use aggregate form features: `last_5`, `last_10`, `season_avg`, minutes aggregates, and deltas versus season average or last-10 form.
- Sequence models use LSTM inputs built from recent game-level deltas and static season context at time `t`.
- Targets are restored next-five-game averages: `target_next_5_pts_avg`, `target_next_5_ast_avg`, and `target_next_5_reb_avg`.
- Evaluation uses temporal train/validation/test splits, with validation used for model selection and test used as the holdout evaluation split.

Task-specific LSTM representation:

- Points: sequence `[game_pts - pts_season_avg_at_game, game_min - min_season_avg_at_game]`, static `[pts_season_avg_at_t, min_season_avg_at_t]`, target delta `next_5_pts_avg - pts_season_avg_at_t`.
- Assists: sequence `[game_ast - ast_season_avg_at_game, game_min - min_season_avg_at_game]`, static `[ast_season_avg_at_t, min_season_avg_at_t]`, target delta `next_5_ast_avg - ast_season_avg_at_t`.
- Rebounds: sequence `[game_reb - reb_season_avg_at_game, game_min - min_season_avg_at_game]`, static `[reb_season_avg_at_t, min_season_avg_at_t]`, target delta `next_5_reb_avg - reb_season_avg_at_t`.

In [ ]:
PERFORMANCE_METADATA_COLS = ["player_id", "as_of_date", "season", "team_id"]
TABULAR_FEATURES = [
    "pts_last_5", "pts_last_10", "pts_season_avg",
    "pts_last_5_minus_season_avg", "pts_last_10_minus_season_avg", "pts_last_5_minus_last_10",
    "ast_last_5", "ast_last_10", "ast_season_avg",
    "ast_last_5_minus_season_avg", "ast_last_10_minus_season_avg", "ast_last_5_minus_last_10",
    "reb_last_5", "reb_last_10", "reb_season_avg",
    "reb_last_5_minus_season_avg", "reb_last_10_minus_season_avg", "reb_last_5_minus_last_10",
    "min_last_5", "min_last_10", "min_season_avg",
    "min_last_5_minus_season_avg", "min_last_10_minus_season_avg", "min_last_5_minus_last_10",
]
PERFORMANCE_TARGETS = {
    "points": "target_next_5_pts_avg",
    "assists": "target_next_5_ast_avg",
    "rebounds": "target_next_5_reb_avg",
}
PERFORMANCE_EVAL_SPLITS = ["validation", "test"]
TARGET_SPECIFIC_FEATURES = {
    "points": [
        "pts_last_5", "pts_last_10", "pts_season_avg",
        "pts_last_5_minus_season_avg", "pts_last_10_minus_season_avg", "pts_last_5_minus_last_10",
        "min_last_5", "min_last_10", "min_season_avg",
        "min_last_5_minus_season_avg", "min_last_10_minus_season_avg", "min_last_5_minus_last_10",
    ],
    "assists": [
        "ast_last_5", "ast_last_10", "ast_season_avg",
        "ast_last_5_minus_season_avg", "ast_last_10_minus_season_avg", "ast_last_5_minus_last_10",
        "min_last_5", "min_last_10", "min_season_avg",
        "min_last_5_minus_season_avg", "min_last_10_minus_season_avg", "min_last_5_minus_last_10",
        "pts_season_avg", "pts_last_10",
    ],
    "rebounds": [
        "reb_last_5", "reb_last_10", "reb_season_avg",
        "reb_last_5_minus_season_avg", "reb_last_10_minus_season_avg", "reb_last_5_minus_last_10",
        "min_last_5", "min_last_10", "min_season_avg",
        "min_last_5_minus_season_avg", "min_last_10_minus_season_avg", "min_last_5_minus_last_10",
    ],
}

def try_import_mlflow():
    """Input: none. Output: mlflow module or None when unavailable."""
    try:
        import mlflow  # type: ignore
        return mlflow
    except ImportError:
        print("MLflow is not installed. Run `%pip install -q mlflow` and re-run this section to enable tracking.")
        return None

def configure_mlflow():
    """Input: none. Output: configured mlflow module or None when tracking is unavailable."""
    mlflow = try_import_mlflow()
    if mlflow is None:
        return None
    tracking_uri = f"sqlite:///{MLFLOW_BACKEND_DB}"
    artifact_uri = MLFLOW_ARTIFACT_DIR.as_uri()
    mlflow.set_tracking_uri(tracking_uri)
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    if experiment is None:
        mlflow.create_experiment(MLFLOW_EXPERIMENT_NAME, artifact_location=artifact_uri)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    print(f"MLflow tracking URI: {tracking_uri}")
    print(f"MLflow artifact URI: {artifact_uri}")
    print(f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")
    return mlflow


TRAINING_CACHE_ENABLED = os.getenv("NBA_SCOUT_USE_MLFLOW_CACHE", "true").lower() in {"1", "true", "yes"}
METRIC_CACHE_DIR = MLFLOW_ARTIFACT_DIR / "metric_cache"
METRIC_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def json_safe(value):
    """Input: arbitrary object. Output: JSON-stable representation for experiment signatures."""
    if isinstance(value, dict):
        return {str(key): json_safe(value[key]) for key in sorted(value)}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def settings_signature(payload: dict[str, object]) -> str:
    """Input: settings payload. Output: stable short hash for cache validation."""
    encoded = json.dumps(json_safe(payload), sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


def dataframe_fingerprint(df: pd.DataFrame, split_col: str = "split") -> dict[str, object]:
    """Input: dataframe. Output: compact fingerprint for cache invalidation when rows/splits change."""
    fingerprint: dict[str, object] = {"rows": len(df), "columns": list(df.columns)}
    if split_col in df.columns:
        fingerprint["split_counts"] = df[split_col].value_counts(dropna=False).sort_index().to_dict()
    for column in ["season", "season_label", "anchor_season"]:
        if column in df.columns:
            values = df[column].dropna().astype(str)
            fingerprint[f"{column}_min"] = values.min() if not values.empty else None
            fingerprint[f"{column}_max"] = values.max() if not values.empty else None
            fingerprint[f"{column}_nunique"] = int(values.nunique())
    return fingerprint


def metric_cache_paths(cache_name: str) -> tuple[Path, Path]:
    """Input: cache name. Output: parquet path and JSON signature path."""
    return METRIC_CACHE_DIR / f"{cache_name}.parquet", METRIC_CACHE_DIR / f"{cache_name}.signature.json"


def load_metric_cache(cache_name: str, signature: str) -> pd.DataFrame | None:
    """Input: cache name and expected signature. Output: cached evaluation dataframe or None."""
    if not TRAINING_CACHE_ENABLED:
        return None
    data_path, signature_path = metric_cache_paths(cache_name)
    if not data_path.exists() or not signature_path.exists():
        return None
    cached_signature = json.loads(signature_path.read_text()).get("settings_signature")
    if cached_signature != signature:
        print(f"Metric cache miss for {cache_name}: settings changed.")
        return None
    cached = pd.read_parquet(data_path)
    print(f"Loaded cached metrics for {cache_name}: {data_path} shape={cached.shape}")
    return cached


def save_metric_cache(cache_name: str, signature: str, evaluation_df: pd.DataFrame, payload: dict[str, object]) -> None:
    """Input: cache metadata and evaluation dataframe. Output: persisted metric cache for future notebook runs."""
    if evaluation_df.empty:
        return
    data_path, signature_path = metric_cache_paths(cache_name)
    evaluation_df.to_parquet(data_path, index=False)
    signature_path.write_text(json.dumps({
        "settings_signature": signature,
        "payload": json_safe(payload),
    }, indent=2, sort_keys=True))
    print(f"Saved metric cache for {cache_name}: {data_path} shape={evaluation_df.shape}")

def try_import_lightgbm():
    """Input: none. Output: LGBMRegressor class or None when LightGBM is unavailable."""
    try:
        from lightgbm import LGBMRegressor  # type: ignore
        return LGBMRegressor
    except ImportError:
        print("LightGBM is not installed. Run `%pip install -q lightgbm` to include it in the tabular experiment.")
        return None

def evaluate_regression(y_true: pd.Series | np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Input: actual and predicted values. Output: MAE, RMSE, and R2 metrics."""
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": r2_score(y_true, y_pred),
    }

def log_model_to_mlflow(
    mlflow,
    model_family: str,
    task_name: str,
    estimator,
    features: list[str],
    evaluation_df: pd.DataFrame,
    cache_signature: str | None = None,
) -> None:
    """Input: mlflow module, model metadata, estimator, features, metrics. Output: logged MLflow run."""
    if mlflow is None:
        return
    run_name = f"{model_family}_{task_name}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "project": "nba-scout-assistant",
            "experiment_type": "tabular",
            "model_family": model_family,
            "task": task_name,
            "target": PERFORMANCE_TARGETS[task_name],
        })
        params = estimator.get_params() if hasattr(estimator, "get_params") else {}
        safe_params = {key: value for key, value in params.items() if isinstance(value, (str, int, float, bool, type(None)))}
        run_params = {
            "n_features": len(features),
            "features": ",".join(features),
            "target": PERFORMANCE_TARGETS[task_name],
            **safe_params,
        }
        if cache_signature is not None:
            run_params["settings_signature"] = cache_signature
        mlflow.log_params(run_params)
        for _, row in evaluation_df.iterrows():
            split = row["split"]
            mlflow.log_metric(f"{split}_rows", int(row["rows"]))
            mlflow.log_metric(f"{split}_mae", float(row["mae"]))
            mlflow.log_metric(f"{split}_rmse", float(row["rmse"]))
            mlflow.log_metric(f"{split}_r2", float(row["r2"]))
        input_example = performance_training_clean[features].head(5)
        mlflow.sklearn.log_model(
            estimator,
            name="model",
            input_example=input_example,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )

mlflow_client = configure_mlflow()

def build_tabular_estimators() -> dict[str, object]:
    """Input: none. Output: configured tabular regressors for the experiment."""
    estimators: dict[str, object] = {
        "ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "random_forest": RandomForestRegressor(
            n_estimators=260,
            min_samples_leaf=10,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        ),
        "mlp": Pipeline([
            ("scaler", StandardScaler()),
            ("model", MLPRegressor(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=0.001,
                learning_rate_init=0.0007,
                max_iter=300,
                early_stopping=True,
                n_iter_no_change=12,
                random_state=42,
            )),
        ]),
    }
    LGBMRegressor = try_import_lightgbm()
    if LGBMRegressor is not None:
        estimators["lightgbm"] = LGBMRegressor(
            n_estimators=450,
            learning_rate=0.02,
            num_leaves=15,
            min_child_samples=40,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            verbose=-1,
        )
    return estimators

def train_tabular_models(df: pd.DataFrame) -> tuple[dict[str, object], pd.DataFrame]:
    """Input: clean performance dataframe. Output: fitted tabular models and evaluation table."""
    models = {}
    rows = []
    estimators = build_tabular_estimators()
    for model_name, estimator_template in estimators.items():
        for task_name, target_col in PERFORMANCE_TARGETS.items():
            import copy
            estimator = copy.deepcopy(estimator_template)
            features = TARGET_SPECIFIC_FEATURES[task_name]
            train_df = df[df["split"].eq("train")].dropna(subset=[*features, target_col])
            estimator.fit(train_df[features], train_df[target_col])
            model_key = f"{model_name}_{task_name}"
            models[model_key] = estimator
            for split_name in PERFORMANCE_EVAL_SPLITS:
                eval_df = df[df["split"].eq(split_name)].dropna(subset=[*features, target_col])
                if eval_df.empty:
                    continue
                metrics = evaluate_regression(eval_df[target_col], estimator.predict(eval_df[features]))
                rows.append({
                    "experiment": "tabular",
                    "model": model_name,
                    "task": task_name,
                    "split": split_name,
                    "rows": len(eval_df),
                    "features": ", ".join(features),
                    **metrics,
                })
            task_eval = pd.DataFrame([row for row in rows if row["model"] == model_name and row["task"] == task_name])
            log_model_to_mlflow(mlflow_client, model_name, task_name, estimator, features, task_eval, cache_signature=tabular_cache_signature)
    return models, pd.DataFrame(rows)

tabular_cache_payload = {
    "cache_version": "tabular_v2_delta_features",
    "targets": PERFORMANCE_TARGETS,
    "features": TARGET_SPECIFIC_FEATURES,
    "eval_splits": PERFORMANCE_EVAL_SPLITS,
    "estimators": {name: estimator.get_params() if hasattr(estimator, "get_params") else str(estimator) for name, estimator in build_tabular_estimators().items()},
    "data": dataframe_fingerprint(performance_training_clean),
}
tabular_cache_signature = settings_signature(tabular_cache_payload)
tabular_evaluation = load_metric_cache("tabular_evaluation", tabular_cache_signature)
if tabular_evaluation is not None:
    tabular_models = {}
else:
    tabular_models, tabular_evaluation = train_tabular_models(performance_training_clean)
    save_metric_cache("tabular_evaluation", tabular_cache_signature, tabular_evaluation, tabular_cache_payload)
tabular_evaluation


### LSTM Sequence Models

The LSTM experiment predicts next-five-game production from recent game-level sequences. Each task uses target-specific stat deltas plus minutes deltas, then restores the prediction back to the original stat scale for evaluation.

The model combines two inputs:

- `sequence_delta`: recent game sequence with shape `(sequence_length, 2)`.
- `static_context`: season-to-date stat average and minutes average at the prediction date.

Training uses Huber loss, Adam optimization, early stopping on validation MAE, learning-rate reduction on plateau, and MLflow tracking.

In [ ]:
LSTM_MAX_EPOCHS = 100
LSTM_PATIENCE = 8
LSTM_MIN_DELTA = 1e-4
SHORT_TERM_EVALUATION_PATH = GOLD_DIR / "short_term_model_evaluation.parquet"
SHORT_TERM_SELECTED_SUMMARY_PATH = GOLD_DIR / "short_term_model_selection_summary.parquet"

LSTM_TASK_CONFIG = {
    "points": {
        "stat_col": "pts",
        "stat_avg_col": "pts_season_avg",
        "target_col": "target_next_5_pts_avg",
        "sequence_length": 10,
        "units": 64,
        "batch_size": 128,
        "learning_rate": 5e-4,
        "loss": "huber",
        "dropout": 0.15,
        "scale_target_delta": False,
        "sequence_features": ["pts_delta", "min_delta"],
        "static_features": ["pts_season_avg", "min_season_avg"],
    },
    "assists": {
        "stat_col": "ast",
        "stat_avg_col": "ast_season_avg",
        "target_col": "target_next_5_ast_avg",
        "sequence_length": 10,
        "units": 64,
        "batch_size": 128,
        "learning_rate": 5e-4,
        "loss": "huber",
        "dropout": 0.15,
        "scale_target_delta": False,
        "sequence_features": ["ast_delta", "min_delta"],
        "static_features": ["ast_season_avg", "min_season_avg"],
    },
    "rebounds": {
        "stat_col": "reb",
        "stat_avg_col": "reb_season_avg",
        "target_col": "target_next_5_reb_avg",
        "sequence_length": 15,
        "units": 64,
        "batch_size": 128,
        "learning_rate": 5e-4,
        "loss": "huber",
        "dropout": 0.15,
        "scale_target_delta": False,
        "sequence_features": ["reb_delta", "min_delta"],
        "static_features": ["reb_season_avg", "min_season_avg"],
    },
}


def try_import_tensorflow():
    """Input: none. Output: tensorflow module or None when unavailable."""
    try:
        import tensorflow as tf  # type: ignore
        return tf
    except ImportError:
        print("TensorFlow is not installed. In Colab, run `%pip install -q tensorflow` or enable a runtime with TensorFlow.")
        return None


def prepare_sequence_training(df: pd.DataFrame) -> pd.DataFrame:
    """Input: point-in-time performance table. Output: rows ready for task-specific delta-sequence LSTM construction."""
    sequence_df = df.copy()
    if "split" not in sequence_df.columns:
        sequence_df["split"] = sequence_df["season"].map(assign_temporal_split)

    required_cols = ["player_id", "season", "as_of_date", "game_id", "split", "min", "min_season_avg"]
    for config in LSTM_TASK_CONFIG.values():
        required_cols.extend([config["stat_col"], config["stat_avg_col"], config["target_col"]])
    required_cols = sorted(set(required_cols))

    missing_cols = sorted(set(required_cols) - set(sequence_df.columns))
    if missing_cols:
        raise KeyError(f"Missing LSTM sequence columns: {missing_cols}")

    before = len(sequence_df)
    sequence_df = sequence_df.dropna(subset=required_cols).copy()
    sequence_df = sequence_df.sort_values(["player_id", "season", "as_of_date", "game_id"]).reset_index(drop=True)
    print(f"Dropped rows without LSTM inputs or next-5 targets: {before:,} -> {len(sequence_df):,}")
    return sequence_df


def make_lstm_delta_sequences(
    df: pd.DataFrame,
    task_config: dict[str, object],
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Input: sequence-ready rows and task config. Output: X_seq, X_static, y_delta, y_actual, baseline, split arrays."""
    stat_col = task_config["stat_col"]
    stat_avg_col = task_config["stat_avg_col"]
    target_col = task_config["target_col"]
    sequence_length = int(task_config["sequence_length"])

    rows_x_seq, rows_x_static, rows_y_delta, rows_y_actual, rows_baseline, rows_split = [], [], [], [], [], []
    ordered = df.sort_values(["player_id", "season", "as_of_date", "game_id"]).copy()

    for _, group in ordered.groupby(["player_id", "season"], sort=False):
        stat_delta = group[stat_col].to_numpy(dtype="float32") - group[stat_avg_col].to_numpy(dtype="float32")
        min_delta = group["min"].to_numpy(dtype="float32") - group["min_season_avg"].to_numpy(dtype="float32")
        sequence_values = np.column_stack([stat_delta, min_delta]).astype("float32")
        static_values = group[[stat_avg_col, "min_season_avg"]].to_numpy(dtype="float32")
        targets_actual = group[target_col].to_numpy(dtype="float32")
        baselines = group[stat_avg_col].to_numpy(dtype="float32")
        splits = group["split"].to_numpy()

        for idx in range(sequence_length - 1, len(group)):
            window = sequence_values[idx - sequence_length + 1 : idx + 1]
            static_context = static_values[idx]
            target_actual = targets_actual[idx]
            baseline = baselines[idx]
            target_delta = target_actual - baseline

            if (
                np.isnan(window).any()
                or np.isnan(static_context).any()
                or np.isnan(target_actual)
                or np.isnan(target_delta)
            ):
                continue

            rows_x_seq.append(window)
            rows_x_static.append(static_context)
            rows_y_delta.append(target_delta)
            rows_y_actual.append(target_actual)
            rows_baseline.append(baseline)
            rows_split.append(splits[idx])

    return (
        np.asarray(rows_x_seq, dtype="float32"),
        np.asarray(rows_x_static, dtype="float32"),
        np.asarray(rows_y_delta, dtype="float32"),
        np.asarray(rows_y_actual, dtype="float32"),
        np.asarray(rows_baseline, dtype="float32"),
        np.asarray(rows_split),
    )


def scale_lstm_inputs(
    X_seq: np.ndarray,
    X_static: np.ndarray,
    y_delta: np.ndarray,
    train_mask: np.ndarray,
    scale_target_delta: bool,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, StandardScaler | None, StandardScaler, StandardScaler]:
    """Input: raw sequence/static/delta arrays. Output: scaled inputs and optional fitted target scaler."""
    seq_scaler = StandardScaler()
    static_scaler = StandardScaler()
    n_sequence_features = X_seq.shape[-1]

    seq_scaler.fit(X_seq[train_mask].reshape(-1, n_sequence_features))
    static_scaler.fit(X_static[train_mask])

    X_seq_scaled = seq_scaler.transform(X_seq.reshape(-1, n_sequence_features)).reshape(X_seq.shape).astype("float32")
    X_static_scaled = static_scaler.transform(X_static).astype("float32")

    if scale_target_delta:
        y_scaler = StandardScaler()
        y_scaler.fit(y_delta[train_mask].reshape(-1, 1))
        y_delta_model = y_scaler.transform(y_delta.reshape(-1, 1)).reshape(-1).astype("float32")
    else:
        y_scaler = None
        y_delta_model = y_delta.astype("float32")

    return X_seq_scaled, X_static_scaled, y_delta_model, y_scaler, seq_scaler, static_scaler

def build_lstm_model(tf, task_config: dict[str, object], n_sequence_features: int, n_static_features: int):
    """Input: tensorflow module, task config, input dimensions. Output: compiled selected two-input LSTM delta model."""
    sequence_length = int(task_config["sequence_length"])
    units = int(task_config["units"])
    dropout = float(task_config["dropout"])
    learning_rate = float(task_config["learning_rate"])

    sequence_input = tf.keras.layers.Input(shape=(sequence_length, n_sequence_features), name="sequence_delta")
    sequence_branch = tf.keras.layers.LSTM(units=units, return_sequences=False, name="sequence_encoder")(sequence_input)
    sequence_branch = tf.keras.layers.Dropout(dropout, name="sequence_dropout")(sequence_branch)

    static_input = tf.keras.layers.Input(shape=(n_static_features,), name="static_context")
    static_branch = tf.keras.layers.Dense(16, activation="relu", name="static_encoder")(static_input)

    combined = tf.keras.layers.Concatenate(name="feature_fusion")([sequence_branch, static_branch])
    combined = tf.keras.layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name="fusion_dense",
    )(combined)
    combined = tf.keras.layers.Dropout(0.10, name="fusion_dropout")(combined)
    output = tf.keras.layers.Dense(1, name="predicted_delta_scaled")(combined)

    model = tf.keras.Model(inputs=[sequence_input, static_input], outputs=output)
    if task_config["loss"] == "huber":
        loss = tf.keras.losses.Huber(delta=1.0)
    elif task_config["loss"] == "mse":
        loss = tf.keras.losses.MeanSquaredError()
    else:
        raise ValueError(f"Unsupported LSTM loss: {task_config['loss']}")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss=loss,
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


def log_lstm_model_to_mlflow(
    mlflow,
    model,
    task_name: str,
    task_config: dict[str, object],
    evaluation_df: pd.DataFrame,
    cache_signature: str | None = None,
) -> None:
    """Input: mlflow module, keras model, task metadata, metrics. Output: logged MLflow run."""
    if mlflow is None:
        return
    run_name = f"lstm_delta{task_config['sequence_length']}_{task_name}_u{task_config['units']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "project": "nba-scout-assistant",
            "experiment_type": "short_term_sequence_delta_static_selected",
            "model_family": "lstm",
            "task": task_name,
            "target": task_config["target_col"],
        })
        run_params = {
            "sequence_length": task_config["sequence_length"],
            "max_epochs": LSTM_MAX_EPOCHS,
            "patience": LSTM_PATIENCE,
            "batch_size": task_config["batch_size"],
            "units": task_config["units"],
            "learning_rate": task_config["learning_rate"],
            "loss": task_config["loss"],
            "dropout": task_config["dropout"],
            "scale_target_delta": task_config.get("scale_target_delta", False),
            "sequence_features": ",".join(task_config["sequence_features"]),
            "static_features": ",".join(task_config["static_features"]),
            "target_delta": f"{task_config['target_col']}-{task_config['stat_avg_col']}",
            "restored_prediction": f"{task_config['stat_avg_col']}+inverse_scaled_predicted_delta",
        }
        if cache_signature is not None:
            run_params["settings_signature"] = cache_signature
        mlflow.log_params(run_params)
        for _, row in evaluation_df.iterrows():
            split = row["split"]
            mlflow.log_metric(f"{split}_rows", int(row["rows"]))
            mlflow.log_metric(f"{split}_mae", float(row["mae"]))
            mlflow.log_metric(f"{split}_rmse", float(row["rmse"]))
            mlflow.log_metric(f"{split}_r2", float(row["r2"]))
        mlflow.keras.log_model(model, name="model")


def train_lstm_models(df: pd.DataFrame) -> tuple[dict[str, object], pd.DataFrame]:
    """Input: untrimmed performance dataframe. Output: selected fitted LSTM models and restored-scale evaluation table."""
    tf = try_import_tensorflow()
    if tf is None:
        return {}, pd.DataFrame()

    sequence_training = prepare_sequence_training(df)
    models = {}
    rows = []

    for task_name, task_config in LSTM_TASK_CONFIG.items():
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(42)
        X_seq, X_static, y_delta, y_actual, baseline, splits = make_lstm_delta_sequences(sequence_training, task_config)
        train_mask = splits == "train"
        val_mask = splits == "validation"
        if not train_mask.any() or not val_mask.any():
            print(f"Skipping {task_name}: missing train or validation sequences.")
            continue

        X_seq_scaled, X_static_scaled, y_delta_model, y_scaler, seq_scaler, static_scaler = scale_lstm_inputs(
            X_seq,
            X_static,
            y_delta,
            train_mask,
            scale_target_delta=bool(task_config.get("scale_target_delta", False)),
        )
        model = build_lstm_model(
            tf,
            task_config=task_config,
            n_sequence_features=X_seq.shape[-1],
            n_static_features=X_static.shape[-1],
        )
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_mae",
                mode="min",
                patience=LSTM_PATIENCE,
                min_delta=LSTM_MIN_DELTA,
                restore_best_weights=True,
                verbose=1,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_mae",
                mode="min",
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                verbose=1,
            ),
        ]
        history = model.fit(
            [X_seq_scaled[train_mask], X_static_scaled[train_mask]],
            y_delta_model[train_mask],
            validation_data=([X_seq_scaled[val_mask], X_static_scaled[val_mask]], y_delta_model[val_mask]),
            epochs=LSTM_MAX_EPOCHS,
            batch_size=int(task_config["batch_size"]),
            verbose=1,
            callbacks=callbacks,
        )
        model_key = f"lstm_delta{task_config['sequence_length']}_{task_name}_u{task_config['units']}"
        models[model_key] = {
            "model": model,
            "y_scaler": y_scaler,
            "sequence_scaler": seq_scaler,
            "static_scaler": static_scaler,
            "config": task_config,
        }

        task_rows = []
        for split_name in PERFORMANCE_EVAL_SPLITS:
            eval_mask = splits == split_name
            if not eval_mask.any():
                continue
            pred_delta_scaled = model.predict(
                [X_seq_scaled[eval_mask], X_static_scaled[eval_mask]],
                batch_size=int(task_config["batch_size"]),
                verbose=0,
            ).reshape(-1)
            if y_scaler is not None:
                pred_delta = y_scaler.inverse_transform(pred_delta_scaled.reshape(-1, 1)).reshape(-1)
            else:
                pred_delta = pred_delta_scaled
            pred_actual = baseline[eval_mask] + pred_delta
            metrics = evaluate_regression(y_actual[eval_mask], pred_actual)
            row = {
                "experiment": "sequence_delta_static_selected",
                "model": model_key,
                "task": task_name,
                "split": split_name,
                "rows": int(eval_mask.sum()),
                "features": ", ".join([*task_config["sequence_features"], *task_config["static_features"]]),
                "epochs_ran": len(history.history["loss"]),
                **metrics,
            }
            rows.append(row)
            task_rows.append(row)

        log_lstm_model_to_mlflow(
            mlflow_client,
            model=model,
            task_name=task_name,
            task_config=task_config,
            evaluation_df=pd.DataFrame(task_rows),
            cache_signature=lstm_cache_signature,
        )

    return models, pd.DataFrame(rows)


lstm_cache_payload = {
    "cache_version": "lstm_delta_static_v2",
    "task_config": LSTM_TASK_CONFIG,
    "max_epochs": LSTM_MAX_EPOCHS,
    "patience": LSTM_PATIENCE,
    "min_delta": LSTM_MIN_DELTA,
    "eval_splits": PERFORMANCE_EVAL_SPLITS,
    "data": dataframe_fingerprint(performance_training_clean),
}
lstm_cache_signature = settings_signature(lstm_cache_payload)
lstm_evaluation = load_metric_cache("lstm_evaluation", lstm_cache_signature)
if lstm_evaluation is not None:
    lstm_models = {}
else:
    lstm_models, lstm_evaluation = train_lstm_models(performance_training)
    save_metric_cache("lstm_evaluation", lstm_cache_signature, lstm_evaluation, lstm_cache_payload)
lstm_evaluation


In [ ]:
model_evaluation = pd.concat(
    [frame for frame in [tabular_evaluation, lstm_evaluation] if not frame.empty],
    ignore_index=True,
)

if not model_evaluation.empty:
    model_evaluation.to_parquet(SHORT_TERM_EVALUATION_PATH, index=False)
    print(f"Saved {SHORT_TERM_EVALUATION_PATH} with shape {model_evaluation.shape}")

    short_term_selection_summary = (
        model_evaluation[model_evaluation["split"].eq("validation")]
        .sort_values(["task", "mae"])
        .groupby("task", as_index=False)
        .head(1)
        .loc[:, ["task", "experiment", "model", "rows", "features", "mae", "rmse", "r2"]]
        .rename(columns={"mae": "validation_mae", "rmse": "validation_rmse", "r2": "validation_r2"})
        .reset_index(drop=True)
    )

    test_metrics = model_evaluation[model_evaluation["split"].eq("test")][["task", "experiment", "model", "mae", "rmse", "r2"]].rename(
        columns={"mae": "test_mae", "rmse": "test_rmse", "r2": "test_r2"}
    )

    short_term_selection_summary = short_term_selection_summary.merge(
        test_metrics,
        on=["task", "experiment", "model"],
        how="left",
    )
    short_term_selection_summary.to_parquet(SHORT_TERM_SELECTED_SUMMARY_PATH, index=False)
    print(f"Saved {SHORT_TERM_SELECTED_SUMMARY_PATH} with shape {short_term_selection_summary.shape}")

    display(short_term_selection_summary)
    display(model_evaluation.sort_values(["task", "split", "mae"]).reset_index(drop=True))
else:
    print("No model evaluation rows were produced.")


### Short-Term Model Selection Notes

Short-term model selection is based on validation MAE and reported again on the temporal test split.

Current LSTM configuration:

- max epochs `100`
- batch size `128`
- Huber loss
- Adam learning rate `5e-4`
- gradient clipping `clipnorm=1.0`
- EarlyStopping on `val_mae` with patience `8`
- ReduceLROnPlateau on `val_mae`
- direct delta prediction without target scaling

Task-specific sequence settings:

| Task | Sequence length | Units | Sequence features | Static features |
|---|---:|---:|---|---|
| points | 10 | 64 | `pts_delta`, `min_delta` | `pts_season_avg`, `min_season_avg` |
| assists | 10 | 64 | `ast_delta`, `min_delta` | `ast_season_avg`, `min_season_avg` |
| rebounds | 15 | 64 | `reb_delta`, `min_delta` | `reb_season_avg`, `min_season_avg` |

The tabular MLP, tree-based models, and Ridge baselines remain in the comparison set. The selected short-term summary table chooses the winner by validation MAE for each task.

## 9. Long-Term Player Trajectory Forecasting

This section builds a season-anchor dataset for direct multi-horizon player forecasting. The goal is to estimate future availability and per-minute production for scouting and roster decisions.

Each row represents a player at a season-end anchor. Features use information available through that anchor: recent form, lagged season summaries, age, career mileage, position, workload trends, and role indicators.

Primary modeling targets are created for `h=1`, `h=2`, and `h=3` future seasons:

- active probability
- points per 36 minutes
- assists per 36 minutes
- rebounds per 36 minutes

Availability-risk labels are derived from future participation counts for each forecast horizon.

In [ ]:
LONG_TERM_OUTPUT_PATH = GOLD_DIR / "long_term_player_forecast_training.parquet"
LONG_TERM_HORIZONS = [1, 2, 3]
LONG_TERM_CAREER_LAGS = 4
LONG_TERM_RECENT_GAMES = 20
NBA_REGULAR_SEASON_GAMES = 82


def season_label_to_start_year(value: object) -> int | None:
    """Input: NBA season label. Output: season start year integer or None."""
    if pd.isna(value):
        return None
    match = re.search(r"(\d{4})", str(value))
    return int(match.group(1)) if match else None


def safe_per_36(numerator: pd.Series, minutes: pd.Series) -> pd.Series:
    """Input: counting stat and minutes. Output: per-36 rate with zero-minute rows set to NaN."""
    minutes = pd.to_numeric(minutes, errors="coerce")
    numerator = pd.to_numeric(numerator, errors="coerce")
    return np.where(minutes.gt(0), numerator / minutes * 36, np.nan)


def slope_last_values(values: pd.Series, min_points: int = 2) -> float:
    """Input: ordered numeric values. Output: simple linear slope over non-missing recent values."""
    clean = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype="float64")
    if len(clean) < min_points:
        return np.nan
    x = np.arange(len(clean), dtype="float64")
    return float(np.polyfit(x, clean, 1)[0])


def build_player_season_summary(game_logs_df: pd.DataFrame, players_df: pd.DataFrame, season_stats_df: pd.DataFrame) -> pd.DataFrame:
    """Input: game logs, players, season stats. Output: player-season summary table for long-term anchors and targets."""
    if game_logs_df.empty:
        return pd.DataFrame()

    logs = game_logs_df.copy()
    logs["game_date"] = pd.to_datetime(logs["game_date"])
    numeric_cols = [
        "minutes", "points", "assists", "rebounds", "fga", "fta", "fg3a",
        "true_shooting_pct", "rest_days",
    ]
    for column in numeric_cols:
        if column in logs.columns:
            logs[column] = pd.to_numeric(logs[column], errors="coerce")

    grouped = logs.groupby(["player_id", "season"], sort=False)
    summary = grouped.agg(
        player_name=("player_name", "last"),
        team_id=("team_id", "last"),
        season_end_date=("game_date", "max"),
        games_played=("game_id", "nunique"),
        total_minutes=("minutes", "sum"),
        minutes_per_game=("minutes", "mean"),
        points=("points", "sum"),
        assists=("assists", "sum"),
        rebounds=("rebounds", "sum"),
        fga=("fga", "sum"),
        fta=("fta", "sum"),
        fg3a=("fg3a", "sum"),
        true_shooting_pct=("true_shooting_pct", "mean"),
        rest_days_avg=("rest_days", "mean"),
    ).reset_index()

    summary["season_start_year"] = summary["season"].map(season_label_to_start_year).astype("Int64")
    summary["availability_rate"] = (summary["games_played"] / NBA_REGULAR_SEASON_GAMES).clip(0, 1)
    summary["pts_per_36"] = safe_per_36(summary["points"], summary["total_minutes"])
    summary["ast_per_36"] = safe_per_36(summary["assists"], summary["total_minutes"])
    summary["reb_per_36"] = safe_per_36(summary["rebounds"], summary["total_minutes"])
    summary["fga_per_36"] = safe_per_36(summary["fga"], summary["total_minutes"])
    summary["fta_per_36"] = safe_per_36(summary["fta"], summary["total_minutes"])
    summary["fg3a_per_36"] = safe_per_36(summary["fg3a"], summary["total_minutes"])

    if not season_stats_df.empty:
        advanced_cols = [
            col for col in [
                "player_id", "season", "usage_pct", "pace", "offensive_rating", "defensive_rating"
            ]
            if col in season_stats_df.columns
        ]
        if {"player_id", "season"}.issubset(advanced_cols):
            summary = summary.merge(
                season_stats_df[advanced_cols].drop_duplicates(["player_id", "season"]),
                on=["player_id", "season"],
                how="left",
            )

    player_cols = [col for col in ["player_id", "birth_date", "position", "height", "weight"] if col in players_df.columns]
    if player_cols:
        player_meta = players_df[player_cols].drop_duplicates("player_id").copy()
        summary = summary.merge(player_meta, on="player_id", how="left")

    if "birth_date" in summary.columns:
        birth_date = pd.to_datetime(summary["birth_date"], errors="coerce")
        summary["age_at_anchor"] = (summary["season_end_date"] - birth_date).dt.days / 365.25
    elif "age" in summary.columns:
        summary["age_at_anchor"] = pd.to_numeric(summary["age"], errors="coerce")
    else:
        summary["age_at_anchor"] = np.nan

    summary = summary.sort_values(["player_id", "season_start_year"]).reset_index(drop=True)
    summary["career_games"] = summary.groupby("player_id")["games_played"].cumsum()
    summary["career_minutes"] = summary.groupby("player_id")["total_minutes"].cumsum()
    summary["years_in_league"] = summary.groupby("player_id").cumcount() + 1
    return summary


def build_recent_game_anchor_features(game_logs_df: pd.DataFrame, recent_games: int = LONG_TERM_RECENT_GAMES) -> pd.DataFrame:
    """Input: game logs. Output: recent-form features from the last N games before each season-end anchor."""
    if game_logs_df.empty:
        return pd.DataFrame()

    logs = game_logs_df.copy()
    logs["game_date"] = pd.to_datetime(logs["game_date"])
    for column in ["minutes", "points", "assists", "rebounds", "fga", "fta", "true_shooting_pct", "rest_days"]:
        if column in logs.columns:
            logs[column] = pd.to_numeric(logs[column], errors="coerce")

    recent_rows = []
    for (player_id, season), group in logs.sort_values("game_date").groupby(["player_id", "season"], sort=False):
        recent = group.tail(recent_games).copy()
        total_minutes = recent["minutes"].sum()
        row = {
            "player_id": player_id,
            "season": season,
            "recent_games_count": len(recent),
            "recent_minutes_per_game": recent["minutes"].mean(),
            "recent_true_shooting_pct": recent["true_shooting_pct"].mean() if "true_shooting_pct" in recent.columns else np.nan,
            "recent_rest_days_avg": recent["rest_days"].mean() if "rest_days" in recent.columns else np.nan,
            "recent_minutes_trend": slope_last_values(recent["minutes"]),
            "recent_pts_trend": slope_last_values(recent["points"]),
            "recent_ast_trend": slope_last_values(recent["assists"]),
            "recent_reb_trend": slope_last_values(recent["rebounds"]),
        }
        for stat, output in [("points", "recent_pts_per_36"), ("assists", "recent_ast_per_36"), ("rebounds", "recent_reb_per_36"), ("fga", "recent_fga_per_36"), ("fta", "recent_fta_per_36")]:
            if stat in recent.columns:
                row[output] = float(recent[stat].sum() / total_minutes * 36) if total_minutes > 0 else np.nan
        recent_rows.append(row)

    return pd.DataFrame(recent_rows)


def add_lagged_season_features(anchor_df: pd.DataFrame, career_lags: int = LONG_TERM_CAREER_LAGS) -> pd.DataFrame:
    """Input: season summary rows. Output: anchor rows with flattened current/prior season features and trends."""
    feature_cols = [
        "age_at_anchor", "games_played", "minutes_per_game", "total_minutes",
        "availability_rate", "pts_per_36", "ast_per_36", "reb_per_36",
        "fga_per_36", "fta_per_36", "true_shooting_pct", "usage_pct",
    ]
    feature_cols = [col for col in feature_cols if col in anchor_df.columns]
    rows = []

    ordered = anchor_df.sort_values(["player_id", "season_start_year"]).copy()
    for _, group in ordered.groupby("player_id", sort=False):
        group = group.reset_index(drop=True)
        for idx, anchor in group.iterrows():
            row = {
                "player_id": anchor["player_id"],
                "player_name": anchor.get("player_name"),
                "anchor_season": anchor["season"],
                "anchor_season_start_year": anchor["season_start_year"],
                "anchor_date": anchor["season_end_date"],
                "team_id": anchor.get("team_id"),
                "position": anchor.get("position", "UNK"),
                "height": anchor.get("height"),
                "weight": anchor.get("weight"),
                "age_at_anchor": anchor.get("age_at_anchor"),
                "years_in_league": anchor.get("years_in_league"),
                "career_games": anchor.get("career_games"),
                "career_minutes": anchor.get("career_minutes"),
            }
            history = group.loc[:idx].tail(career_lags).reset_index(drop=True)
            for lag in range(career_lags):
                hist_idx = len(history) - 1 - lag
                for feature in feature_cols:
                    row[f"{feature}_lag_{lag}"] = history.loc[hist_idx, feature] if hist_idx >= 0 else np.nan

            trend_window = history.tail(3)
            for feature in ["games_played", "minutes_per_game", "pts_per_36", "ast_per_36", "reb_per_36", "total_minutes"]:
                if feature in trend_window.columns:
                    row[f"{feature}_slope_3yr"] = slope_last_values(trend_window[feature])
            rows.append(row)

    return pd.DataFrame(rows)


def add_future_horizon_targets(anchor_features: pd.DataFrame, season_summary: pd.DataFrame, horizons: list[int] = LONG_TERM_HORIZONS) -> pd.DataFrame:
    """Input: anchor features and season summary. Output: long-term targets for observed future seasons."""
    result = anchor_features.copy()
    future_lookup = season_summary.set_index(["player_id", "season_start_year"])
    max_observed_season_start = int(season_summary["season_start_year"].max())

    def empty_target(horizon: int, active_value: float | int | None, games_value: float | int | None) -> dict[str, object]:
        return {
            f"active_h{horizon}": active_value,
            f"games_played_h{horizon}": games_value,
            f"minutes_per_game_h{horizon}": np.nan,
            f"pts_per_36_h{horizon}": np.nan,
            f"ast_per_36_h{horizon}": np.nan,
            f"reb_per_36_h{horizon}": np.nan,
            f"pts_per_game_h{horizon}": np.nan,
            f"ast_per_game_h{horizon}": np.nan,
            f"reb_per_game_h{horizon}": np.nan,
        }

    for horizon in horizons:
        target_rows = []
        for _, row in result[["player_id", "anchor_season_start_year"]].iterrows():
            future_season_start = row["anchor_season_start_year"] + horizon
            key = (row["player_id"], future_season_start)
            if key in future_lookup.index:
                target = future_lookup.loc[key]
                if isinstance(target, pd.DataFrame):
                    target = target.iloc[0]
                active = int(pd.notna(target.get("games_played")) and target.get("games_played", 0) > 0)
                target_rows.append({
                    f"active_h{horizon}": active,
                    f"games_played_h{horizon}": target.get("games_played"),
                    f"minutes_per_game_h{horizon}": target.get("minutes_per_game"),
                    f"pts_per_36_h{horizon}": target.get("pts_per_36"),
                    f"ast_per_36_h{horizon}": target.get("ast_per_36"),
                    f"reb_per_36_h{horizon}": target.get("reb_per_36"),
                    f"pts_per_game_h{horizon}": target.get("minutes_per_game") * target.get("pts_per_36") / 36 if pd.notna(target.get("minutes_per_game")) and pd.notna(target.get("pts_per_36")) else np.nan,
                    f"ast_per_game_h{horizon}": target.get("minutes_per_game") * target.get("ast_per_36") / 36 if pd.notna(target.get("minutes_per_game")) and pd.notna(target.get("ast_per_36")) else np.nan,
                    f"reb_per_game_h{horizon}": target.get("minutes_per_game") * target.get("reb_per_36") / 36 if pd.notna(target.get("minutes_per_game")) and pd.notna(target.get("reb_per_36")) else np.nan,
                })
            elif future_season_start <= max_observed_season_start:
                target_rows.append(empty_target(horizon, active_value=0, games_value=0))
            else:
                target_rows.append(empty_target(horizon, active_value=np.nan, games_value=np.nan))
        result = pd.concat([result.reset_index(drop=True), pd.DataFrame(target_rows)], axis=1)
        result[f"age_at_h{horizon}"] = result["age_at_anchor"] + horizon

    return result


def build_long_term_training(game_logs_df: pd.DataFrame, players_df: pd.DataFrame, season_stats_df: pd.DataFrame) -> pd.DataFrame:
    """Input: project player/game/season data. Output: season-anchor long-term forecast training dataset."""
    season_summary = build_player_season_summary(game_logs_df, players_df, season_stats_df)
    recent_features = build_recent_game_anchor_features(game_logs_df)
    anchor_features = add_lagged_season_features(season_summary)

    if not recent_features.empty:
        anchor_features = anchor_features.merge(
            recent_features,
            left_on=["player_id", "anchor_season"],
            right_on=["player_id", "season"],
            how="left",
        ).drop(columns=["season"], errors="ignore")

    long_term = add_future_horizon_targets(anchor_features, season_summary)
    required_future_cols = [f"active_h{horizon}" for horizon in LONG_TERM_HORIZONS]
    before = len(long_term)
    long_term = long_term.dropna(subset=required_future_cols).copy()
    long_term["split"] = long_term["anchor_season"].map(assign_long_term_temporal_split)
    long_term = long_term[long_term["split"].isin(["train", "validation", "test"])].copy()
    print(f"Dropped long-term anchors without full h{max(LONG_TERM_HORIZONS)} target coverage: {before:,} -> {len(long_term):,}")
    return long_term.sort_values(["anchor_season_start_year", "player_id"]).reset_index(drop=True)


long_term_training = build_long_term_training(game_logs, players, season_stats)
long_term_training.to_parquet(LONG_TERM_OUTPUT_PATH, index=False)
print(f"Saved {LONG_TERM_OUTPUT_PATH} with shape {long_term_training.shape}")
long_term_training.head()


In [ ]:
LONG_TERM_METADATA_COLS = [
    "player_id", "player_name", "anchor_season", "anchor_season_start_year", "anchor_date", "team_id", "split",
]
LONG_TERM_ANALYSIS_TARGETS = [
    f"active_h{h}" for h in LONG_TERM_HORIZONS
] + [
    f"{target}_h{h}"
    for h in LONG_TERM_HORIZONS
    for target in ["pts_per_36", "ast_per_36", "reb_per_36"]
]


def summarize_long_term_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Input: long-term training dataframe. Output: target coverage and basic distribution summary."""
    rows = []
    for column in LONG_TERM_ANALYSIS_TARGETS:
        if column not in df.columns:
            continue
        values = pd.to_numeric(df[column], errors="coerce")
        rows.append({
            "target": column,
            "non_null_rows": int(values.notna().sum()),
            "missing_pct": float(values.isna().mean()),
            "mean": float(values.mean()) if values.notna().any() else np.nan,
            "std": float(values.std()) if values.notna().any() else np.nan,
            "p10": float(values.quantile(0.10)) if values.notna().any() else np.nan,
            "p50": float(values.quantile(0.50)) if values.notna().any() else np.nan,
            "p90": float(values.quantile(0.90)) if values.notna().any() else np.nan,
        })
    return pd.DataFrame(rows)


def summarize_long_term_feature_missingness(df: pd.DataFrame) -> pd.DataFrame:
    """Input: long-term training dataframe. Output: high-missing feature summary for audit."""
    excluded = set(LONG_TERM_ANALYSIS_TARGETS + LONG_TERM_METADATA_COLS)
    feature_cols = [
        column
        for column in df.columns
        if column not in excluded and not str(column).endswith(("_h1", "_h2", "_h3"))
    ]
    summary = (
        df[feature_cols]
        .isna()
        .mean()
        .rename("missing_pct")
        .reset_index()
        .rename(columns={"index": "feature"})
        .sort_values("missing_pct", ascending=False)
    )
    return summary


def top_numeric_correlations(df: pd.DataFrame, target_col: str, top_n: int = 15) -> pd.DataFrame:
    """Input: dataframe and target. Output: strongest absolute numeric correlations for quick feature analysis."""
    if target_col not in df.columns:
        return pd.DataFrame()
    excluded = set(LONG_TERM_ANALYSIS_TARGETS + LONG_TERM_METADATA_COLS)
    numeric_features = [
        column for column in df.columns
        if column not in excluded and pd.api.types.is_numeric_dtype(df[column])
    ]
    corr_rows = []
    target = pd.to_numeric(df[target_col], errors="coerce")
    for feature in numeric_features:
        pair = pd.concat([pd.to_numeric(df[feature], errors="coerce"), target], axis=1).dropna()
        if len(pair) < 50 or pair.iloc[:, 0].nunique() <= 1:
            continue
        corr = pair.iloc[:, 0].corr(pair.iloc[:, 1])
        corr_rows.append({"feature": feature, "target": target_col, "correlation": corr, "abs_correlation": abs(corr)})
    return pd.DataFrame(corr_rows).sort_values("abs_correlation", ascending=False).head(top_n)


long_term_target_summary = summarize_long_term_targets(long_term_training)
long_term_missing_summary = summarize_long_term_feature_missingness(long_term_training)
long_term_feature_correlations = pd.concat(
    [
        top_numeric_correlations(long_term_training, target_col)
        for target_col in ["active_h1", "pts_per_36_h1", "ast_per_36_h1", "reb_per_36_h1"]
    ],
    ignore_index=True,
)

display(long_term_target_summary)
display(long_term_missing_summary.head(25))
display(long_term_feature_correlations)


In [ ]:
LONG_TERM_TARGETS = {
    horizon: {
        "active_probability": f"active_h{horizon}",
        "pts_per_36": f"pts_per_36_h{horizon}",
        "ast_per_36": f"ast_per_36_h{horizon}",
        "reb_per_36": f"reb_per_36_h{horizon}",
    }
    for horizon in LONG_TERM_HORIZONS
}
LONG_TERM_REGRESSION_TARGETS = ["pts_per_36", "ast_per_36", "reb_per_36"]
LONG_TERM_EVALUATION_PATH = GOLD_DIR / "long_term_model_evaluation.parquet"
LONG_TERM_SELECTION_SUMMARY_PATH = GOLD_DIR / "long_term_model_selection_summary.parquet"
LONG_TERM_EVAL_SPLITS = ["validation", "test"]
LONG_TERM_PRIMARY_METRIC = {
    "active_probability": "brier",
    "pts_per_36": "mae",
    "ast_per_36": "mae",
    "reb_per_36": "mae",
}
LONG_TERM_SELECTED_MODEL_REGISTRY = {
    ("active_probability", 1): "lightgbm_classifier",
    ("active_probability", 2): "logistic_age_curve",
    ("active_probability", 3): "logistic_age_curve",
    ("pts_per_36", 1): "random_forest",
    ("pts_per_36", 2): "random_forest",
    ("pts_per_36", 3): "ridge_age_curve",
    ("ast_per_36", 1): "hist_gradient_boosting",
    ("ast_per_36", 2): "ridge_age_curve",
    ("ast_per_36", 3): "ridge_age_curve",
    ("reb_per_36", 1): "lightgbm_regressor",
    ("reb_per_36", 2): "random_forest",
    ("reb_per_36", 3): "ridge_age_curve",
}


def try_import_lightgbm_estimators():
    """Input: none. Output: LightGBM classifier/regressor classes or (None, None) when unavailable."""
    try:
        from lightgbm import LGBMClassifier, LGBMRegressor  # type: ignore
        return LGBMClassifier, LGBMRegressor
    except ImportError:
        print("LightGBM is not installed. Run `%pip install -q lightgbm` to enable long-term boosting models.")
        return None, None


def get_one_hot_encoder():
    """Input: none. Output: OneHotEncoder compatible with local sklearn version."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def is_future_horizon_column(column: str) -> bool:
    """Input: column name. Output: True when it is a future horizon field that would leak target-period data."""
    return str(column).endswith(("_h1", "_h2", "_h3"))


def long_term_excluded_columns(active_horizon: int, columns: pd.Index | list[str] | None = None) -> set[str]:
    """Input: forecast horizon and dataframe columns. Output: columns excluded from model features to prevent target leakage."""
    excluded = set(LONG_TERM_METADATA_COLS)
    for target_group in LONG_TERM_TARGETS.values():
        excluded.update(target_group.values())
    candidate_columns = list(columns) if columns is not None else list(long_term_training.columns)
    excluded.update(column for column in candidate_columns if is_future_horizon_column(column))
    return excluded


def prepare_long_term_raw_matrix(
    df: pd.DataFrame,
    target_col: str,
    horizon: int,
    require_active_target: str | None = None,
) -> tuple[pd.DataFrame, pd.Series, pd.Series, list[str]]:
    """Input: long-term data, target, horizon, optional active filter. Output: raw X, y, split labels, feature names."""
    model_df = df.copy()
    if require_active_target is not None:
        model_df = model_df[model_df[require_active_target].eq(1)].copy()
    model_df = model_df.dropna(subset=[target_col]).copy()

    feature_cols = [column for column in model_df.columns if column not in long_term_excluded_columns(horizon, model_df.columns)]
    X = model_df[feature_cols].copy()
    y = pd.to_numeric(model_df[target_col], errors="coerce")
    splits = model_df["split"].copy()
    return X, y, splits, feature_cols


def build_long_term_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Input: raw feature matrix. Output: sklearn preprocessor for numeric and categorical columns."""
    categorical_cols = [column for column in X.columns if X[column].dtype == "object"]
    numeric_cols = [column for column in X.columns if column not in categorical_cols]
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
            ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("one_hot", get_one_hot_encoder())]), categorical_cols),
        ],
        remainder="drop",
    )


def evaluate_binary(y_true: pd.Series, y_score: np.ndarray) -> dict[str, float]:
    """Input: binary labels and probability scores. Output: Brier score and ROC-AUC where available."""
    metrics = {"brier": float(brier_score_loss(y_true, y_score))}
    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_score))
    except ValueError:
        metrics["roc_auc"] = np.nan
    return metrics


def build_long_term_baseline_model(task_name: str, X: pd.DataFrame) -> Pipeline:
    """Input: target task and raw X. Output: interpretable baseline model pipeline."""
    if task_name == "active_probability":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5)),
        ])
    return Pipeline([
        ("preprocess", build_long_term_preprocessor(X)),
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=5.0)),
    ])


def build_long_term_selected_model(task_name: str, horizon: int, model_key: str, X: pd.DataFrame) -> Pipeline:
    """Input: task, horizon, selected model key, raw X. Output: strongest selected model pipeline."""
    LGBMClassifier, LGBMRegressor = try_import_lightgbm_estimators()
    preprocessor = build_long_term_preprocessor(X)

    if model_key in {"logistic_age_curve", "logistic_age_curve_provisional", "ridge_age_curve_provisional"}:
        return build_long_term_baseline_model(task_name, X)
    if model_key == "ridge_age_curve":
        return build_long_term_baseline_model(task_name, X)
    if model_key == "lightgbm_classifier":
        if LGBMClassifier is None:
            return build_long_term_baseline_model(task_name, X)
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=31, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1)),
        ])
    if model_key == "lightgbm_regressor":
        if LGBMRegressor is None:
            return Pipeline([("preprocess", preprocessor), ("model", HistGradientBoostingRegressor(max_iter=250, learning_rate=0.04, l2_regularization=0.05, random_state=42))])
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", LGBMRegressor(n_estimators=350, learning_rate=0.03, num_leaves=31, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1)),
        ])
    if model_key == "random_forest":
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=8, max_features="sqrt", n_jobs=-1, random_state=42)),
        ])
    if model_key == "hist_gradient_boosting":
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", HistGradientBoostingRegressor(max_iter=250, learning_rate=0.04, l2_regularization=0.05, random_state=42)),
        ])
    raise ValueError(f"Unsupported selected model key: {model_key}")


def clip_long_term_prediction(task_name: str, predictions: np.ndarray) -> np.ndarray:
    """Input: target name and predictions. Output: predictions clipped to plausible NBA bounds."""
    clipped = np.asarray(predictions, dtype="float64")
    if task_name.endswith("per_36"):
        return np.clip(clipped, 0, None)
    return clipped


def evaluate_long_term_model(
    model: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    splits: pd.Series,
    task_name: str,
    model_label: str,
    model_family: str,
    horizon: int,
    feature_cols: list[str],
) -> list[dict[str, object]]:
    """Input: fitted model and evaluation data. Output: split-level metric rows."""
    rows = []
    for split_name in LONG_TERM_EVAL_SPLITS:
        eval_mask = splits.eq(split_name)
        if not eval_mask.any():
            continue
        if task_name == "active_probability":
            y_eval = y[eval_mask].astype(int)
            if y_eval.nunique() < 2:
                continue
            pred = model.predict_proba(X[eval_mask])[:, 1]
            metrics = evaluate_binary(y_eval, pred)
        else:
            pred = clip_long_term_prediction(task_name, model.predict(X[eval_mask]))
            metrics = evaluate_regression(y[eval_mask], pred)
        rows.append({
            "experiment": "long_term_model_selection",
            "model_family": model_family,
            "model": model_label,
            "horizon": horizon,
            "target": task_name,
            "split": split_name,
            "rows": int(eval_mask.sum()),
            "feature_count": len(feature_cols),
            "features": ", ".join(feature_cols),
            **metrics,
        })
    return rows


def train_long_term_baseline_and_selected_models(df: pd.DataFrame) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame]:
    """Input: long-term training data. Output: fitted baseline/selected models, metrics, and selection summary."""
    models = {}
    rows = []

    for horizon, target_group in LONG_TERM_TARGETS.items():
        for task_name, target_col in target_group.items():
            require_active = None if task_name == "active_probability" else target_group["active_probability"]
            X, y, splits, feature_cols = prepare_long_term_raw_matrix(
                df=df,
                target_col=target_col,
                horizon=horizon,
                require_active_target=require_active,
            )
            train_mask = splits.eq("train")
            if not train_mask.any():
                continue
            if task_name == "active_probability" and y[train_mask].nunique() < 2:
                continue

            baseline_model = build_long_term_baseline_model(task_name, X)
            baseline_model.fit(X[train_mask], y[train_mask])
            baseline_key = "logistic_age_curve" if task_name == "active_probability" else "ridge_age_curve"
            models[f"baseline_{baseline_key}_{task_name}_h{horizon}"] = baseline_model
            rows.extend(evaluate_long_term_model(
                model=baseline_model,
                X=X,
                y=y,
                splits=splits,
                task_name=task_name,
                model_label=baseline_key,
                model_family="baseline",
                horizon=horizon,
                feature_cols=feature_cols,
            ))

            selected_key = LONG_TERM_SELECTED_MODEL_REGISTRY[(task_name, horizon)]
            selected_model = build_long_term_selected_model(task_name, horizon, selected_key, X)
            selected_model.fit(X[train_mask], y[train_mask])
            models[f"selected_{selected_key}_{task_name}_h{horizon}"] = selected_model
            rows.extend(evaluate_long_term_model(
                model=selected_model,
                X=X,
                y=y,
                splits=splits,
                task_name=task_name,
                model_label=selected_key,
                model_family="selected_strongest",
                horizon=horizon,
                feature_cols=feature_cols,
            ))

    evaluation = pd.DataFrame(rows)
    if evaluation.empty:
        return models, evaluation, pd.DataFrame()

    summary_rows = []
    for (target, horizon), group in evaluation.groupby(["target", "horizon"], sort=False):
        primary_metric = LONG_TERM_PRIMARY_METRIC[target]
        validation_rows = group[group["split"].eq("validation")].copy()
        if validation_rows.empty:
            summary_rows.append({
                "target": target,
                "horizon": horizon,
                "selected_by": "no_validation_target_available",
                "recommended_model": LONG_TERM_SELECTED_MODEL_REGISTRY[(target, horizon)],
                "primary_metric": primary_metric,
                "validation_metric": np.nan,
                "test_metric": np.nan,
            })
            continue
        best = validation_rows.sort_values(primary_metric).iloc[0]
        test_rows = group[
            group["split"].eq("test")
            & group["model"].eq(best["model"])
            & group["model_family"].eq(best["model_family"])
        ]
        summary_rows.append({
            "target": target,
            "horizon": horizon,
            "selected_by": "validation",
            "recommended_model": best["model"],
            "model_family": best["model_family"],
            "primary_metric": primary_metric,
            "validation_metric": best[primary_metric],
            "test_metric": test_rows.iloc[0][primary_metric] if not test_rows.empty and primary_metric in test_rows.columns else np.nan,
            "validation_rows": best["rows"],
            "test_rows": test_rows.iloc[0]["rows"] if not test_rows.empty else np.nan,
        })

    existing_pairs = {(row["target"], row["horizon"]) for row in summary_rows}
    for (target, horizon), model_key in LONG_TERM_SELECTED_MODEL_REGISTRY.items():
        if (target, horizon) in existing_pairs:
            continue
        summary_rows.append({
            "target": target,
            "horizon": horizon,
            "selected_by": "validation",
            "recommended_model": model_key,
            "model_family": "selected_strongest",
            "primary_metric": LONG_TERM_PRIMARY_METRIC[target],
            "validation_metric": np.nan,
            "test_metric": np.nan,
            "validation_rows": 0,
            "test_rows": 0,
        })

    selection_summary = pd.DataFrame(summary_rows).sort_values(["horizon", "target"]).reset_index(drop=True)
    return models, evaluation.sort_values(["horizon", "target", "split", "model_family"]).reset_index(drop=True), selection_summary


long_term_cache_payload = {
    "cache_version": "long_term_primary_targets_v2",
    "horizons": LONG_TERM_HORIZONS,
    "eval_splits": LONG_TERM_EVAL_SPLITS,
    "selected_model_registry": LONG_TERM_SELECTED_MODEL_REGISTRY,
    "primary_metric": LONG_TERM_PRIMARY_METRIC,
    "data": dataframe_fingerprint(long_term_training, split_col="split"),
}
long_term_cache_signature = settings_signature(long_term_cache_payload)
long_term_evaluation = load_metric_cache("long_term_evaluation", long_term_cache_signature)
long_term_selection_summary = load_metric_cache("long_term_selection_summary", long_term_cache_signature)
if long_term_evaluation is not None and long_term_selection_summary is not None:
    long_term_models = {}
else:
    long_term_models, long_term_evaluation, long_term_selection_summary = train_long_term_baseline_and_selected_models(long_term_training)
    save_metric_cache("long_term_evaluation", long_term_cache_signature, long_term_evaluation, long_term_cache_payload)
    save_metric_cache("long_term_selection_summary", long_term_cache_signature, long_term_selection_summary, long_term_cache_payload)

if not long_term_evaluation.empty:
    long_term_evaluation.to_parquet(LONG_TERM_EVALUATION_PATH, index=False)
    print(f"Saved {LONG_TERM_EVALUATION_PATH} with shape {long_term_evaluation.shape}")

if not long_term_selection_summary.empty:
    long_term_selection_summary.to_parquet(LONG_TERM_SELECTION_SUMMARY_PATH, index=False)
    print(f"Saved {LONG_TERM_SELECTION_SUMMARY_PATH} with shape {long_term_selection_summary.shape}")

display(long_term_selection_summary)
display(long_term_evaluation)


### Long-Term Optional Per-Game Production Composition

Long-term production is modeled primarily as per-36 output: `pts_per_36`, `ast_per_36`, and `reb_per_36`. Per-game production is treated as an optional derived output because future minutes per game can be noisy.

The notebook evaluates the optional per-game path in two layers:

- Intrinsic minutes quality: predicted future `minutes_per_game` versus actual future `minutes_per_game`.
- Downstream production quality: `predicted_per_36 * predicted_minutes_per_game / 36` versus actual future per-game production.

Per-game output is only recommended when it beats the direct per-game baseline and the minutes estimator passes the configured quality thresholds. Otherwise, the product should keep per-36 production as the long-term output.


In [ ]:
LONG_TERM_PER_GAME_EVALUATION_PATH = GOLD_DIR / "long_term_per_game_composition_evaluation.parquet"
LONG_TERM_DIRECT_PER_GAME_BASELINE_PATH = GOLD_DIR / "long_term_direct_per_game_baseline_evaluation.parquet"
LONG_TERM_PER_GAME_SELECTION_PATH = GOLD_DIR / "long_term_per_game_composition_selection_summary.parquet"
LONG_TERM_PER_GAME_DECISION_PATH = GOLD_DIR / "long_term_per_game_output_decision_summary.parquet"
LONG_TERM_PER_GAME_STATS = ["pts", "ast", "reb"]
LONG_TERM_MINUTES_MAX_MAE_FOR_OUTPUT = 4.0
LONG_TERM_MINUTES_MIN_R2_FOR_OUTPUT = 0.50
LONG_TERM_COMPOSE_MAX_DIRECT_AVG_MAE_GAP = 0.0
LONG_TERM_MINUTES_ESTIMATOR_REGISTRY = {
    1: ["last_mpg", "weighted_lag_mpg", "age_decay_weighted_mpg", "ridge_mpg", "random_forest_mpg", "hist_gradient_boosting_mpg", "lightgbm_mpg"],
    2: ["last_mpg", "weighted_lag_mpg", "age_decay_weighted_mpg", "ridge_mpg", "random_forest_mpg", "hist_gradient_boosting_mpg", "lightgbm_mpg"],
    3: ["last_mpg", "weighted_lag_mpg", "age_decay_weighted_mpg", "ridge_mpg", "random_forest_mpg", "hist_gradient_boosting_mpg", "lightgbm_mpg"],
}


def long_term_model_feature_columns(df: pd.DataFrame, horizon: int) -> list[str]:
    """Input: long-term data and horizon. Output: non-leaking model feature columns."""
    excluded = long_term_excluded_columns(horizon, df.columns)
    feature_cols = [column for column in df.columns if column not in excluded]
    return [column for column in feature_cols if not pd.api.types.is_datetime64_any_dtype(df[column])]


def build_minutes_heuristic_predictions(df: pd.DataFrame, estimator_key: str) -> pd.Series:
    """Input: long-term data and heuristic key. Output: estimated future minutes per game."""
    empty = pd.Series(np.nan, index=df.index, dtype="float64")
    lag0 = pd.to_numeric(df.get("minutes_per_game_lag_0", empty), errors="coerce")
    lag1 = pd.to_numeric(df.get("minutes_per_game_lag_1", empty), errors="coerce")
    lag2 = pd.to_numeric(df.get("minutes_per_game_lag_2", empty), errors="coerce")
    if estimator_key == "last_mpg":
        return lag0
    weighted = (0.60 * lag0) + (0.30 * lag1.fillna(lag0)) + (0.10 * lag2.fillna(lag1).fillna(lag0))
    if estimator_key == "weighted_lag_mpg":
        return weighted
    if estimator_key == "age_decay_weighted_mpg":
        age = pd.to_numeric(df.get("age_at_anchor"), errors="coerce")
        age_penalty = np.where(age.gt(30), 1 - ((age - 30).clip(lower=0, upper=8) * 0.025), 1.0)
        return weighted * age_penalty
    raise ValueError(f"Unsupported minutes heuristic: {estimator_key}")


def build_minutes_model(estimator_key: str, X: pd.DataFrame) -> Pipeline | None:
    """Input: estimator key and raw features. Output: sklearn pipeline for future minutes per game."""
    if estimator_key == "ridge_mpg":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("scale", StandardScaler()),
            ("model", Ridge(alpha=10.0)),
        ])
    if estimator_key == "random_forest_mpg":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", RandomForestRegressor(
                n_estimators=350,
                min_samples_leaf=8,
                max_features=0.55,
                n_jobs=-1,
                random_state=42,
            )),
        ])
    if estimator_key == "hist_gradient_boosting_mpg":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", HistGradientBoostingRegressor(
                max_iter=250,
                learning_rate=0.04,
                l2_regularization=0.05,
                random_state=42,
            )),
        ])
    if estimator_key == "lightgbm_mpg":
        _, LGBMRegressor = try_import_lightgbm_estimators()
        if LGBMRegressor is None:
            return None
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", LGBMRegressor(
                n_estimators=500,
                learning_rate=0.025,
                num_leaves=31,
                min_child_samples=35,
                subsample=0.90,
                colsample_bytree=0.80,
                reg_alpha=0.05,
                reg_lambda=0.25,
                random_state=42,
                verbosity=-1,
            )),
        ])
    return None


def build_direct_per_game_model(model_key: str, X: pd.DataFrame) -> Pipeline | None:
    """Input: model key and raw features. Output: direct per-game baseline model pipeline."""
    if model_key == "ridge":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("scale", StandardScaler()),
            ("model", Ridge(alpha=10.0)),
        ])
    if model_key == "random_forest":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", RandomForestRegressor(
                n_estimators=350,
                min_samples_leaf=8,
                max_features=0.55,
                n_jobs=-1,
                random_state=42,
            )),
        ])
    if model_key == "hist_gradient_boosting":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", HistGradientBoostingRegressor(
                max_iter=250,
                learning_rate=0.04,
                l2_regularization=0.05,
                random_state=42,
            )),
        ])
    if model_key == "lightgbm":
        _, LGBMRegressor = try_import_lightgbm_estimators()
        if LGBMRegressor is None:
            return None
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", LGBMRegressor(
                n_estimators=500,
                learning_rate=0.025,
                num_leaves=31,
                min_child_samples=35,
                subsample=0.90,
                colsample_bytree=0.80,
                reg_alpha=0.05,
                reg_lambda=0.25,
                random_state=42,
                verbosity=-1,
            )),
        ])
    raise ValueError(f"Unsupported direct per-game model: {model_key}")


def log_long_term_per_game_metrics_to_mlflow(
    mlflow,
    evaluation_df: pd.DataFrame,
    direct_baseline_df: pd.DataFrame,
    decision_df: pd.DataFrame,
    cache_signature: str | None = None,
) -> None:
    """Input: mlflow module and evaluation tables. Output: one MLflow run with per-game composition metrics."""
    if mlflow is None:
        return
    with mlflow.start_run(run_name="long_term_per_game_composition"):
        mlflow.set_tags({
            "project": "nba-scout-assistant",
            "experiment_type": "long_term_per_game_composition",
            "metric_objective": "downstream_per_game_mae",
        })
        if cache_signature is not None:
            mlflow.log_param("cache_signature", cache_signature)
        mlflow.log_param("horizons", ",".join(map(str, LONG_TERM_HORIZONS)))
        mlflow.log_param("stats", ",".join(LONG_TERM_PER_GAME_STATS))
        if not evaluation_df.empty:
            for row in evaluation_df.itertuples(index=False):
                prefix = f"compose_h{row.horizon}_{row.split}_{row.minutes_estimator}"
                mlflow.log_metric(f"{prefix}_avg_pg_mae", float(row.avg_pg_mae))
                for stat in LONG_TERM_PER_GAME_STATS:
                    value = getattr(row, f"{stat}_pg_mae")
                    if pd.notna(value):
                        mlflow.log_metric(f"{prefix}_{stat}_pg_mae", float(value))
        if not direct_baseline_df.empty:
            best_direct = direct_baseline_df.sort_values("mae").groupby(["horizon", "target", "split"], as_index=False).first()
            for row in best_direct.itertuples(index=False):
                metric_name = f"direct_h{row.horizon}_{row.split}_{row.target}_{row.model}_mae"
                mlflow.log_metric(metric_name, float(row.mae))
        if not decision_df.empty:
            for row in decision_df.itertuples(index=False):
                prefix = f"decision_h{row.horizon}_{row.split}"
                if pd.notna(row.compose_avg_pg_mae):
                    mlflow.log_metric(f"{prefix}_compose_avg_pg_mae", float(row.compose_avg_pg_mae))
                if pd.notna(row.direct_avg_pg_mae):
                    mlflow.log_metric(f"{prefix}_direct_avg_pg_mae", float(row.direct_avg_pg_mae))
                if pd.notna(row.minutes_mae):
                    mlflow.log_metric(f"{prefix}_minutes_mae", float(row.minutes_mae))
                mlflow.log_param(f"{prefix}_recommendation", row.recommendation)


def train_and_evaluate_long_term_per_game_composition(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Input: long-term training data. Output: composed per-game evaluation, direct baseline evaluation, and selection summary."""
    rate_predictions: dict[tuple[int, str, str], pd.Series] = {}
    minutes_predictions: dict[tuple[int, str, str], pd.Series] = {}
    composition_rows: list[dict[str, object]] = []
    direct_rows: list[dict[str, object]] = []

    for horizon in LONG_TERM_HORIZONS:
        feature_cols = long_term_model_feature_columns(df, horizon)
        X_all = df[feature_cols].copy()
        splits = df["split"].copy()

        for stat in LONG_TERM_PER_GAME_STATS:
            rate_task = f"{stat}_per_36"
            rate_target = f"{rate_task}_h{horizon}"
            if rate_target not in df.columns:
                continue
            rate_train_mask = splits.eq("train") & df[rate_target].notna()
            if not rate_train_mask.any():
                continue
            rate_model_key = LONG_TERM_SELECTED_MODEL_REGISTRY.get((rate_task, horizon), "ridge_age_curve")
            rate_model = build_long_term_selected_model(rate_task, horizon, rate_model_key, X_all)
            rate_model.fit(X_all[rate_train_mask], pd.to_numeric(df.loc[rate_train_mask, rate_target], errors="coerce"))
            for split_name in LONG_TERM_EVAL_SPLITS:
                eval_mask = splits.eq(split_name) & df[rate_target].notna()
                if eval_mask.any():
                    rate_predictions[(horizon, stat, split_name)] = pd.Series(
                        rate_model.predict(X_all[eval_mask]),
                        index=df.index[eval_mask],
                    )

            direct_target = f"{stat}_per_game_h{horizon}"
            if direct_target in df.columns:
                direct_train_mask = splits.eq("train") & df[direct_target].notna()
                for model_key in ["ridge", "random_forest", "hist_gradient_boosting", "lightgbm"]:
                    if not direct_train_mask.any():
                        continue
                    direct_model = build_direct_per_game_model(model_key, X_all)
                    if direct_model is None:
                        continue
                    direct_model.fit(X_all[direct_train_mask], pd.to_numeric(df.loc[direct_train_mask, direct_target], errors="coerce"))
                    for split_name in LONG_TERM_EVAL_SPLITS:
                        eval_mask = splits.eq(split_name) & df[direct_target].notna()
                        if not eval_mask.any():
                            continue
                        y_true = pd.to_numeric(df.loc[eval_mask, direct_target], errors="coerce")
                        y_pred = direct_model.predict(X_all[eval_mask])
                        metrics = evaluate_regression(y_true, y_pred)
                        direct_rows.append({
                            "experiment": "long_term_direct_per_game_baseline",
                            "model": model_key,
                            "target": direct_target,
                            "horizon": horizon,
                            "split": split_name,
                            "rows": int(eval_mask.sum()),
                            **metrics,
                        })

        minutes_target = f"minutes_per_game_h{horizon}"
        for estimator_key in LONG_TERM_MINUTES_ESTIMATOR_REGISTRY[horizon]:
            if estimator_key in {"last_mpg", "weighted_lag_mpg", "age_decay_weighted_mpg"}:
                for split_name in LONG_TERM_EVAL_SPLITS:
                    eval_mask = splits.eq(split_name)
                    minutes_predictions[(horizon, estimator_key, split_name)] = build_minutes_heuristic_predictions(df.loc[eval_mask], estimator_key)
                continue
            if minutes_target not in df.columns:
                continue
            minutes_train_mask = splits.eq("train") & df[minutes_target].notna()
            if not minutes_train_mask.any():
                continue
            minutes_model = build_minutes_model(estimator_key, X_all)
            if minutes_model is None:
                continue
            minutes_model.fit(X_all[minutes_train_mask], pd.to_numeric(df.loc[minutes_train_mask, minutes_target], errors="coerce"))
            for split_name in LONG_TERM_EVAL_SPLITS:
                eval_mask = splits.eq(split_name) & df[minutes_target].notna()
                if eval_mask.any():
                    minutes_predictions[(horizon, estimator_key, split_name)] = pd.Series(
                        minutes_model.predict(X_all[eval_mask]).clip(min=0, max=48),
                        index=df.index[eval_mask],
                    )

        for estimator_key in LONG_TERM_MINUTES_ESTIMATOR_REGISTRY[horizon]:
            for split_name in LONG_TERM_EVAL_SPLITS:
                estimator_prediction = minutes_predictions.get((horizon, estimator_key, split_name))
                if estimator_prediction is None or estimator_prediction.empty:
                    continue
                row = {
                    "experiment": "long_term_per_game_composition",
                    "minutes_estimator": estimator_key,
                    "horizon": horizon,
                    "split": split_name,
                    "rows": int(len(estimator_prediction)),
                }
                minutes_target = f"minutes_per_game_h{horizon}"
                if minutes_target in df.columns:
                    minutes_index = estimator_prediction.index[df.loc[estimator_prediction.index, minutes_target].notna()]
                    if len(minutes_index) > 0:
                        minutes_metrics = evaluate_regression(
                            pd.to_numeric(df.loc[minutes_index, minutes_target], errors="coerce"),
                            estimator_prediction.loc[minutes_index],
                        )
                        row["minutes_mae"] = minutes_metrics["mae"]
                        row["minutes_rmse"] = minutes_metrics["rmse"]
                        row["minutes_r2"] = minutes_metrics["r2"]
                        row["minutes_rows"] = int(len(minutes_index))
                    else:
                        row["minutes_mae"] = np.nan
                        row["minutes_rmse"] = np.nan
                        row["minutes_r2"] = np.nan
                        row["minutes_rows"] = 0
                else:
                    row["minutes_mae"] = np.nan
                    row["minutes_rmse"] = np.nan
                    row["minutes_r2"] = np.nan
                    row["minutes_rows"] = 0
                stat_maes = []
                for stat in LONG_TERM_PER_GAME_STATS:
                    true_col = f"{stat}_per_game_h{horizon}"
                    rate_pred = rate_predictions.get((horizon, stat, split_name))
                    if true_col not in df.columns or rate_pred is None:
                        row[f"{stat}_pg_mae"] = np.nan
                        row[f"{stat}_pg_r2"] = np.nan
                        row[f"{stat}_rows"] = 0
                        continue
                    common_index = estimator_prediction.index.intersection(rate_pred.index)
                    valid_mask = df.loc[common_index, true_col].notna()
                    eval_index = common_index[valid_mask]
                    if len(eval_index) == 0:
                        row[f"{stat}_pg_mae"] = np.nan
                        row[f"{stat}_pg_r2"] = np.nan
                        row[f"{stat}_rows"] = 0
                        continue
                    y_true = pd.to_numeric(df.loc[eval_index, true_col], errors="coerce")
                    y_pred = rate_pred.loc[eval_index] * estimator_prediction.loc[eval_index] / 36
                    metrics = evaluate_regression(y_true, y_pred)
                    row[f"{stat}_pg_mae"] = metrics["mae"]
                    row[f"{stat}_pg_rmse"] = metrics["rmse"]
                    row[f"{stat}_pg_r2"] = metrics["r2"]
                    row[f"{stat}_rows"] = int(len(eval_index))
                    stat_maes.append(metrics["mae"])
                row["avg_pg_mae"] = float(np.mean(stat_maes)) if stat_maes else np.nan
                composition_rows.append(row)

    composition = pd.DataFrame(composition_rows)
    direct_baseline = pd.DataFrame(direct_rows)
    if composition.empty:
        return composition, direct_baseline, pd.DataFrame()

    selection_rows = []
    for horizon in LONG_TERM_HORIZONS:
        for split_name in LONG_TERM_EVAL_SPLITS:
            split_rows = composition[
                composition["horizon"].eq(horizon)
                & composition["split"].eq(split_name)
                & composition["avg_pg_mae"].notna()
            ]
            if split_rows.empty:
                continue
            best = split_rows.sort_values("avg_pg_mae").iloc[0]
            selection_rows.append({
                "horizon": horizon,
                "split": split_name,
                "selected_by": "lowest_downstream_avg_pg_mae",
                "recommended_minutes_estimator": best["minutes_estimator"],
                "avg_pg_mae": best["avg_pg_mae"],
                "pts_pg_mae": best.get("pts_pg_mae", np.nan),
                "ast_pg_mae": best.get("ast_pg_mae", np.nan),
                "reb_pg_mae": best.get("reb_pg_mae", np.nan),
                "minutes_mae": best.get("minutes_mae", np.nan),
                "minutes_r2": best.get("minutes_r2", np.nan),
                "minutes_rows": best.get("minutes_rows", 0),
                "rows": best["rows"],
            })
    selection = pd.DataFrame(selection_rows).sort_values(["horizon", "split"]).reset_index(drop=True)
    return composition.sort_values(["horizon", "split", "avg_pg_mae"]).reset_index(drop=True), direct_baseline, selection


def build_long_term_per_game_decision_summary(
    composition_df: pd.DataFrame,
    direct_baseline_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: composed and direct per-game evaluations. Output: recommendation for whether per-game output should be used."""
    if composition_df.empty:
        return pd.DataFrame()
    selected = (
        composition_df[composition_df["avg_pg_mae"].notna()]
        .sort_values("avg_pg_mae")
        .groupby(["horizon", "split"], as_index=False)
        .first()
    )
    direct_rows = []
    if not direct_baseline_df.empty:
        direct_best = (
            direct_baseline_df.sort_values("mae")
            .groupby(["horizon", "split", "target"], as_index=False)
            .first()
        )
        for (horizon, split_name), group in direct_best.groupby(["horizon", "split"], sort=False):
            row = {"horizon": horizon, "split": split_name}
            stat_maes = []
            for stat in LONG_TERM_PER_GAME_STATS:
                target = f"{stat}_per_game_h{horizon}"
                target_rows = group[group["target"].eq(target)]
                if target_rows.empty:
                    row[f"direct_{stat}_pg_mae"] = np.nan
                    continue
                mae = float(target_rows.iloc[0]["mae"])
                row[f"direct_{stat}_pg_mae"] = mae
                stat_maes.append(mae)
            row["direct_avg_pg_mae"] = float(np.mean(stat_maes)) if stat_maes else np.nan
            direct_rows.append(row)
    direct_summary = pd.DataFrame(direct_rows)
    summary = selected.merge(direct_summary, on=["horizon", "split"], how="left")
    summary = summary.rename(columns={
        "minutes_estimator": "recommended_minutes_estimator",
        "avg_pg_mae": "compose_avg_pg_mae",
    })
    summary["compose_minus_direct_avg_mae"] = summary["compose_avg_pg_mae"] - summary["direct_avg_pg_mae"]
    summary["minutes_passes_threshold"] = (
        summary["minutes_mae"].le(LONG_TERM_MINUTES_MAX_MAE_FOR_OUTPUT)
        & summary["minutes_r2"].ge(LONG_TERM_MINUTES_MIN_R2_FOR_OUTPUT)
    )
    summary["compose_beats_direct"] = summary["compose_minus_direct_avg_mae"].le(LONG_TERM_COMPOSE_MAX_DIRECT_AVG_MAE_GAP)
    summary["use_per_game_output"] = summary["minutes_passes_threshold"] & summary["compose_beats_direct"]
    summary["recommendation"] = np.where(summary["use_per_game_output"], "optional_per_game", "use_per36_only")
    output_cols = [
        "horizon", "split", "recommendation", "use_per_game_output", "recommended_minutes_estimator",
        "compose_avg_pg_mae", "direct_avg_pg_mae", "compose_minus_direct_avg_mae",
        "minutes_mae", "minutes_r2", "minutes_rows", "minutes_passes_threshold", "compose_beats_direct",
        "pts_pg_mae", "direct_pts_pg_mae", "ast_pg_mae", "direct_ast_pg_mae", "reb_pg_mae", "direct_reb_pg_mae",
    ]
    return summary[[column for column in output_cols if column in summary.columns]].sort_values(["horizon", "split"]).reset_index(drop=True)


per_game_cache_payload = {
    "cache_version": "long_term_per_game_composition_v3_optional_output_decision",
    "horizons": LONG_TERM_HORIZONS,
    "eval_splits": LONG_TERM_EVAL_SPLITS,
    "minutes_estimators": LONG_TERM_MINUTES_ESTIMATOR_REGISTRY,
    "rate_model_registry": LONG_TERM_SELECTED_MODEL_REGISTRY,
    "data": dataframe_fingerprint(long_term_training, split_col="split"),
}
per_game_cache_signature = settings_signature(per_game_cache_payload)
long_term_per_game_evaluation = load_metric_cache("long_term_per_game_composition_evaluation", per_game_cache_signature)
long_term_direct_per_game_baseline = load_metric_cache("long_term_direct_per_game_baseline", per_game_cache_signature)
long_term_per_game_selection = load_metric_cache("long_term_per_game_composition_selection", per_game_cache_signature)
long_term_per_game_decision = load_metric_cache("long_term_per_game_output_decision", per_game_cache_signature)

if long_term_per_game_evaluation is None or long_term_direct_per_game_baseline is None or long_term_per_game_selection is None or long_term_per_game_decision is None:
    long_term_per_game_evaluation, long_term_direct_per_game_baseline, long_term_per_game_selection = train_and_evaluate_long_term_per_game_composition(long_term_training)
    long_term_per_game_decision = build_long_term_per_game_decision_summary(long_term_per_game_evaluation, long_term_direct_per_game_baseline)
    save_metric_cache("long_term_per_game_composition_evaluation", per_game_cache_signature, long_term_per_game_evaluation, per_game_cache_payload)
    save_metric_cache("long_term_direct_per_game_baseline", per_game_cache_signature, long_term_direct_per_game_baseline, per_game_cache_payload)
    save_metric_cache("long_term_per_game_composition_selection", per_game_cache_signature, long_term_per_game_selection, per_game_cache_payload)
    save_metric_cache("long_term_per_game_output_decision", per_game_cache_signature, long_term_per_game_decision, per_game_cache_payload)
    log_long_term_per_game_metrics_to_mlflow(
        mlflow_client,
        long_term_per_game_evaluation,
        long_term_direct_per_game_baseline,
        long_term_per_game_decision,
        cache_signature=per_game_cache_signature,
    )

if not long_term_per_game_evaluation.empty:
    long_term_per_game_evaluation.to_parquet(LONG_TERM_PER_GAME_EVALUATION_PATH, index=False)
    print(f"Saved {LONG_TERM_PER_GAME_EVALUATION_PATH} with shape {long_term_per_game_evaluation.shape}")

if not long_term_direct_per_game_baseline.empty:
    long_term_direct_per_game_baseline.to_parquet(LONG_TERM_DIRECT_PER_GAME_BASELINE_PATH, index=False)
    print(f"Saved {LONG_TERM_DIRECT_PER_GAME_BASELINE_PATH} with shape {long_term_direct_per_game_baseline.shape}")

if not long_term_per_game_selection.empty:
    long_term_per_game_selection.to_parquet(LONG_TERM_PER_GAME_SELECTION_PATH, index=False)
    print(f"Saved {LONG_TERM_PER_GAME_SELECTION_PATH} with shape {long_term_per_game_selection.shape}")

if not long_term_per_game_decision.empty:
    long_term_per_game_decision.to_parquet(LONG_TERM_PER_GAME_DECISION_PATH, index=False)
    print(f"Saved {LONG_TERM_PER_GAME_DECISION_PATH} with shape {long_term_per_game_decision.shape}")

display(long_term_per_game_decision)
display(long_term_per_game_selection)
display(long_term_per_game_evaluation)
display(long_term_direct_per_game_baseline.sort_values(["horizon", "target", "split", "mae"]))


### Long-Term Availability Risk Models

Availability is modeled as classification because exact future games played has high variance across injuries, aging, role changes, team context, and roster decisions.

Risk targets:

- `low_availability`: player appears in 20 or fewer games at the forecast horizon.
- `high_availability`: player appears in 61 or more games at the forecast horizon.

These risk models provide scouting labels and uncertainty signals alongside `active_probability` and per-36 production forecasts.

In [ ]:
LONG_TERM_RISK_EVALUATION_PATH = GOLD_DIR / "long_term_availability_risk_evaluation.parquet"
LONG_TERM_RISK_SELECTION_PATH = GOLD_DIR / "long_term_availability_risk_selection_summary.parquet"
LONG_TERM_RISK_TARGETS = {
    "low_availability": {
        "label": "<=20 games",
        "predicate": lambda games: games <= 20,
        "selected_models": {
            1: "random_forest",
            2: "logistic_age_curve",
            3: "logistic_age_curve",
        },
    },
    "high_availability": {
        "label": ">=61 games",
        "predicate": lambda games: games >= 61,
        "selected_models": {
            1: "random_forest",
            2: "random_forest",
            3: "logistic_age_curve",
        },
    },
}


def add_availability_risk_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Input: long-term training dataframe. Output: copy with binary availability risk labels per horizon."""
    result = df.copy()
    for horizon in LONG_TERM_HORIZONS:
        games_col = f"games_played_h{horizon}"
        if games_col not in result.columns:
            continue
        games = pd.to_numeric(result[games_col], errors="coerce")
        for target_name, target_config in LONG_TERM_RISK_TARGETS.items():
            result[f"{target_name}_h{horizon}"] = np.where(
                games.notna(),
                target_config["predicate"](games).astype(int),
                np.nan,
            )
    return result


def prepare_availability_risk_matrix(
    df: pd.DataFrame,
    target_col: str,
    horizon: int,
) -> tuple[pd.DataFrame, pd.Series, pd.Series, list[str]]:
    """Input: long-term data, risk target, horizon. Output: raw X, y, splits, and feature names without leakage."""
    model_df = df.dropna(subset=[target_col]).copy()
    excluded = long_term_excluded_columns(horizon, model_df.columns)
    for risk_name in LONG_TERM_RISK_TARGETS:
        for risk_horizon in LONG_TERM_HORIZONS:
            excluded.add(f"{risk_name}_h{risk_horizon}")
    excluded.update(column for column in model_df.columns if is_future_horizon_column(column))
    feature_cols = [column for column in model_df.columns if column not in excluded]
    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int)
    splits = model_df["split"].copy()
    return X, y, splits, feature_cols


def build_availability_risk_model(model_key: str, X: pd.DataFrame) -> Pipeline:
    """Input: selected risk model key and raw features. Output: sklearn classifier pipeline."""
    if model_key in {"logistic_age_curve", "logistic_age_curve_provisional"}:
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5)),
        ])
    if model_key == "random_forest":
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=8,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=-1,
                random_state=42,
            )),
        ])
    if model_key == "lightgbm_classifier":
        LGBMClassifier, _ = try_import_lightgbm_estimators()
        if LGBMClassifier is None:
            return build_availability_risk_model("random_forest", X)
        return Pipeline([
            ("preprocess", build_long_term_preprocessor(X)),
            ("model", LGBMClassifier(
                n_estimators=250,
                learning_rate=0.03,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.9,
                colsample_bytree=0.9,
                class_weight="balanced",
                random_state=42,
                verbosity=-1,
            )),
        ])
    raise ValueError(f"Unsupported availability risk model: {model_key}")


def evaluate_availability_risk_model(
    model: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    splits: pd.Series,
    target_name: str,
    horizon: int,
    model_key: str,
    feature_cols: list[str],
) -> list[dict[str, object]]:
    """Input: fitted risk model and split data. Output: split-level risk classification metrics."""
    rows = []
    for split_name in LONG_TERM_EVAL_SPLITS:
        eval_mask = splits.eq(split_name)
        if not eval_mask.any() or y[eval_mask].nunique() < 2:
            continue
        probability = model.predict_proba(X[eval_mask])[:, 1]
        prediction = (probability >= 0.5).astype(int)
        rows.append({
            "experiment": "long_term_availability_risk",
            "model": model_key,
            "target": target_name,
            "horizon": horizon,
            "split": split_name,
            "rows": int(eval_mask.sum()),
            "positive_rate": float(y[eval_mask].mean()),
            "feature_count": len(feature_cols),
            "features": ", ".join(feature_cols),
            "brier": float(brier_score_loss(y[eval_mask], probability)),
            "roc_auc": float(roc_auc_score(y[eval_mask], probability)),
            "precision": float(precision_score(y[eval_mask], prediction, zero_division=0)),
            "recall": float(recall_score(y[eval_mask], prediction, zero_division=0)),
            "f1": float(f1_score(y[eval_mask], prediction, zero_division=0)),
        })
    return rows


def log_availability_risk_model_to_mlflow(
    mlflow,
    model: Pipeline,
    target_name: str,
    horizon: int,
    model_key: str,
    feature_cols: list[str],
    evaluation_df: pd.DataFrame,
    cache_signature: str | None = None,
) -> None:
    """Input: mlflow module, fitted model, metadata, metrics. Output: logged MLflow run."""
    if mlflow is None:
        return
    run_name = f"availability_risk_{target_name}_h{horizon}_{model_key}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "project": "nba-scout-assistant",
            "experiment_type": "long_term_availability_risk",
            "model_family": model_key,
            "target": target_name,
            "horizon": horizon,
        })
        run_params = {
            "target_definition": LONG_TERM_RISK_TARGETS[target_name]["label"],
            "model": model_key,
            "horizon": horizon,
            "feature_count": len(feature_cols),
            "features": ",".join(feature_cols),
            "selection_metric": "validation_brier",
        }
        if cache_signature is not None:
            run_params["settings_signature"] = cache_signature
        mlflow.log_params(run_params)
        for _, row in evaluation_df.iterrows():
            split = row["split"]
            for metric in ["rows", "positive_rate", "brier", "roc_auc", "precision", "recall", "f1"]:
                mlflow.log_metric(f"{split}_{metric}", float(row[metric]))
        mlflow.sklearn.log_model(
            model,
            name="model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )


def train_availability_risk_models(df: pd.DataFrame) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame]:
    """Input: long-term training data. Output: fitted risk models, metrics, and selected-model summary."""
    risk_df = add_availability_risk_targets(df)
    models = {}
    rows = []
    summary_rows = []

    for target_name, target_config in LONG_TERM_RISK_TARGETS.items():
        for horizon in LONG_TERM_HORIZONS:
            target_col = f"{target_name}_h{horizon}"
            model_key = target_config["selected_models"][horizon]
            X, y, splits, feature_cols = prepare_availability_risk_matrix(risk_df, target_col, horizon)
            train_mask = splits.eq("train")
            if not train_mask.any() or y[train_mask].nunique() < 2:
                summary_rows.append({
                    "target": target_name,
                    "horizon": horizon,
                    "selected_by": "no_train_target_available",
                    "recommended_model": model_key,
                    "validation_brier": np.nan,
                    "validation_roc_auc": np.nan,
                    "test_brier": np.nan,
                    "test_roc_auc": np.nan,
                })
                continue

            model = build_availability_risk_model(model_key, X)
            model.fit(X[train_mask], y[train_mask])
            model_name = f"{model_key}_{target_name}_h{horizon}"
            models[model_name] = model

            task_rows = evaluate_availability_risk_model(
                model=model,
                X=X,
                y=y,
                splits=splits,
                target_name=target_name,
                horizon=horizon,
                model_key=model_key,
                feature_cols=feature_cols,
            )
            rows.extend(task_rows)

            task_eval = pd.DataFrame(task_rows)
            if task_eval.empty:
                summary_rows.append({
                    "target": target_name,
                    "horizon": horizon,
                    "selected_by": "no_eval_split_available",
                    "recommended_model": model_key,
                    "validation_brier": np.nan,
                    "validation_roc_auc": np.nan,
                    "test_brier": np.nan,
                    "test_roc_auc": np.nan,
                    "validation_rows": 0,
                    "test_rows": 0,
                })
                continue

            validation = task_eval[task_eval["split"].eq("validation")]
            test = task_eval[task_eval["split"].eq("test")]
            summary_rows.append({
                "target": target_name,
                "horizon": horizon,
                "selected_by": "validation" if not validation.empty else "provisional_no_validation_target_available",
                "recommended_model": model_key,
                "validation_brier": validation.iloc[0]["brier"] if not validation.empty else np.nan,
                "validation_roc_auc": validation.iloc[0]["roc_auc"] if not validation.empty else np.nan,
                "test_brier": test.iloc[0]["brier"] if not test.empty else np.nan,
                "test_roc_auc": test.iloc[0]["roc_auc"] if not test.empty else np.nan,
                "validation_rows": validation.iloc[0]["rows"] if not validation.empty else 0,
                "test_rows": test.iloc[0]["rows"] if not test.empty else 0,
            })

            log_availability_risk_model_to_mlflow(
                mlflow_client,
                model=model,
                target_name=target_name,
                horizon=horizon,
                model_key=model_key,
                feature_cols=feature_cols,
                evaluation_df=task_eval,
                cache_signature=availability_risk_cache_signature,
            )

    evaluation = pd.DataFrame(rows)
    summary = pd.DataFrame(summary_rows).sort_values(["horizon", "target"]).reset_index(drop=True)
    return models, evaluation, summary


availability_risk_cache_payload = {
    "cache_version": "availability_risk_complete_h3_split_v1",
    "risk_targets": {name: {"label": config["label"], "selected_models": config["selected_models"]} for name, config in LONG_TERM_RISK_TARGETS.items()},
    "horizons": LONG_TERM_HORIZONS,
    "eval_splits": LONG_TERM_EVAL_SPLITS,
    "data": dataframe_fingerprint(long_term_training, split_col="split"),
}
availability_risk_cache_signature = settings_signature(availability_risk_cache_payload)
availability_risk_evaluation = load_metric_cache("availability_risk_evaluation", availability_risk_cache_signature)
availability_risk_selection_summary = load_metric_cache("availability_risk_selection_summary", availability_risk_cache_signature)
if availability_risk_evaluation is not None and availability_risk_selection_summary is not None:
    availability_risk_models = {}
else:
    availability_risk_models, availability_risk_evaluation, availability_risk_selection_summary = train_availability_risk_models(long_term_training)
    save_metric_cache("availability_risk_evaluation", availability_risk_cache_signature, availability_risk_evaluation, availability_risk_cache_payload)
    save_metric_cache("availability_risk_selection_summary", availability_risk_cache_signature, availability_risk_selection_summary, availability_risk_cache_payload)

if not availability_risk_evaluation.empty:
    availability_risk_evaluation.to_parquet(LONG_TERM_RISK_EVALUATION_PATH, index=False)
    print(f"Saved {LONG_TERM_RISK_EVALUATION_PATH} with shape {availability_risk_evaluation.shape}")

if not availability_risk_selection_summary.empty:
    availability_risk_selection_summary.to_parquet(LONG_TERM_RISK_SELECTION_PATH, index=False)
    print(f"Saved {LONG_TERM_RISK_SELECTION_PATH} with shape {availability_risk_selection_summary.shape}")

display(availability_risk_selection_summary)
display(availability_risk_evaluation)


### Long-Term Model Selection Notes

Long-term model selection uses validation metrics, with the test split reserved for holdout evaluation. To support complete H3 targets, the long-term split is shifted earlier than the short-term split:

- train: `<= 2019-20`
- validation: `2020-21`
- test: `2021-22`

Recommended baselines:

- `ridge_age_curve` for per-36 regression targets.
- `logistic_age_curve` for availability classification.

Selected models by validation metric:

| Target | Horizon | Selected model | Validation metric |
|---|---:|---|---:|
| active_probability | 1 | LightGBM classifier | Brier 0.1042 |
| active_probability | 2 | Logistic age-curve | Brier 0.1496 |
| active_probability | 3 | Logistic age-curve | Brier 0.1628 |
| pts_per_36 | 1 | RandomForest regressor | MAE 2.41 |
| pts_per_36 | 2 | RandomForest regressor | MAE 2.79 |
| pts_per_36 | 3 | Ridge age-curve | MAE 2.94 |
| ast_per_36 | 1 | HistGradientBoosting regressor | MAE 0.82 |
| ast_per_36 | 2 | Ridge age-curve | MAE 0.85 |
| ast_per_36 | 3 | Ridge age-curve | MAE 0.86 |
| reb_per_36 | 1 | LightGBM regressor | MAE 0.95 |
| reb_per_36 | 2 | RandomForest regressor | MAE 1.00 |
| reb_per_36 | 3 | Ridge age-curve | MAE 1.19 |

Long-term outputs are intended for acquisition ranking, replacement-player search, availability risk review, and production-range estimates.

## 10. Salary Valuation Models

This section estimates player salary value as a share of the league salary cap.

Target:

```text
salary_cap_share = salary_usd / salary_cap_usd
```

The model predicts cap share rather than raw USD so that historical salaries are normalized across cap environments. USD error is still reported by converting predicted cap share back through each season's `salary_cap_usd`.

The modeling dataset is restricted to seasons with stronger role and performance feature coverage. The salary split is:

- train: `2016-17` through `2022-23`
- validation: `2023-24`
- test: `2024-25`

Feature analysis is performed before model selection using correlation and mutual information. The selected compact role feature set emphasizes workload, age, usage, and interpretable role dimensions.

In [ ]:
from sklearn.base import clone
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet

SALARY_TARGET = "salary_cap_share"
SALARY_EVALUATION_PATH = GOLD_DIR / "salary_model_evaluation.parquet"
SALARY_SELECTION_SUMMARY_PATH = GOLD_DIR / "salary_model_selection_summary.parquet"
SALARY_FEATURE_IMPORTANCE_PATH = GOLD_DIR / "salary_feature_importance.parquet"
SALARY_EVAL_SPLITS = ["validation", "test"]
SALARY_TRAIN_SEASONS = ["2016-17", "2017-18", "2018-19", "2019-20", "2020-21", "2021-22", "2022-23"]
SALARY_VALIDATION_SEASONS = ["2023-24"]
SALARY_TEST_SEASONS = ["2024-25"]

SALARY_ROLE_COMPACT_FEATURES = [
    "age",
    "minutes",
    "usage_pct",
    "scoring_creation",
    "playmaking",
    "shooting",
    "rim_pressure",
    "rebounding",
    "perimeter_defense",
    "interior_defense",
    "two_way_impact",
    "position",
]
SALARY_CORE_FEATURES = [
    "age",
    "height",
    "weight",
    "minutes",
    "usage_pct",
    "points_per_100",
    "assists_per_100",
    "rebounds_per_100",
    "true_shooting_pct",
    "three_point_attempt_rate",
    "free_throw_rate",
    "turnover_rate",
    "steal_rate",
    "block_rate",
    "defensive_rebound_rate",
    "foul_rate",
    "scoring_creation",
    "playmaking",
    "shooting",
    "rim_pressure",
    "rebounding",
    "perimeter_defense",
    "interior_defense",
    "two_way_impact",
    "position",
]
SALARY_MISSING_FLAG_FEATURES = [
    column
    for column in salary_training_clean.columns
    if column.endswith("_was_missing")
    and column not in {
        "salary_cap_share_was_missing",
        "salary_cap_usd_was_missing",
        "salary_usd_was_missing",
        "target_salary_usd_was_missing",
    }
]
SALARY_FEATURE_SETS = {
    "role_compact": SALARY_ROLE_COMPACT_FEATURES,
    "core": SALARY_CORE_FEATURES,
    "core_plus_missing_flags": SALARY_CORE_FEATURES + SALARY_MISSING_FLAG_FEATURES,
}
SALARY_SELECTED_FEATURE_SET = "role_compact"
SALARY_SELECTED_MODEL = "lightgbm"


def assign_salary_temporal_split(season_label: object) -> str:
    """Input: season label. Output: salary modeling split for modern salary valuation."""
    if season_label in SALARY_TRAIN_SEASONS:
        return "train"
    if season_label in SALARY_VALIDATION_SEASONS:
        return "validation"
    if season_label in SALARY_TEST_SEASONS:
        return "test"
    return "ignore"


def prepare_salary_modeling_data(df: pd.DataFrame) -> pd.DataFrame:
    """Input: clean salary table. Output: modern salary modeling rows with temporal split."""
    salary_df = df.copy()
    salary_df["split"] = salary_df["season_label"].map(assign_salary_temporal_split)
    salary_df = salary_df[salary_df["split"].isin(["train", "validation", "test"])].copy()
    salary_df = salary_df.dropna(subset=[SALARY_TARGET, "salary_usd", "salary_cap_usd"]).copy()
    salary_df[SALARY_TARGET] = pd.to_numeric(salary_df[SALARY_TARGET], errors="coerce").clip(lower=0)
    salary_df["salary_usd"] = pd.to_numeric(salary_df["salary_usd"], errors="coerce")
    salary_df["salary_cap_usd"] = pd.to_numeric(salary_df["salary_cap_usd"], errors="coerce")
    salary_df = salary_df.dropna(subset=[SALARY_TARGET, "salary_usd", "salary_cap_usd"]).copy()
    return salary_df.reset_index(drop=True)


def build_salary_preprocessor(df: pd.DataFrame, features: list[str], scale_numeric: bool = False) -> ColumnTransformer:
    """Input: modeling dataframe, feature names, scale flag. Output: sklearn preprocessor."""
    numeric_features = [column for column in features if pd.api.types.is_numeric_dtype(df[column])]
    categorical_features = [column for column in features if column not in numeric_features]
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), numeric_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", get_one_hot_encoder()),
        ]), categorical_features),
    ])


def build_salary_estimators() -> dict[str, tuple[object, bool]]:
    """Input: none. Output: candidate salary estimators and whether numeric scaling is needed."""
    estimators: dict[str, tuple[object, bool]] = {
        "ridge": (Ridge(alpha=5.0), True),
        "elasticnet": (ElasticNet(alpha=0.001, l1_ratio=0.2, max_iter=20000, random_state=42), True),
        "random_forest": (RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=8,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        ), False),
        "hist_gradient_boosting": (HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.035,
            l2_regularization=0.05,
            min_samples_leaf=20,
            random_state=42,
        ), False),
        "mlp": (MLPRegressor(
            hidden_layer_sizes=(64, 32),
            alpha=0.001,
            learning_rate_init=0.001,
            early_stopping=True,
            validation_fraction=0.15,
            max_iter=500,
            random_state=42,
        ), True),
    }
    LGBMClassifier, LGBMRegressor = try_import_lightgbm_estimators()
    if LGBMRegressor is not None:
        estimators["lightgbm"] = (LGBMRegressor(
            n_estimators=500,
            learning_rate=0.025,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=42,
            verbosity=-1,
        ), False)
    return estimators


def evaluate_salary_model(model: Pipeline, df: pd.DataFrame, features: list[str]) -> list[dict[str, object]]:
    """Input: fitted model, salary data, feature columns. Output: split-level salary metrics."""
    rows = []
    for split in SALARY_EVAL_SPLITS:
        split_df = df[df["split"].eq(split)].copy()
        if split_df.empty:
            continue
        y_true = split_df[SALARY_TARGET].astype(float)
        pred_share = np.clip(model.predict(split_df[features]), 0, None)
        pred_usd = pred_share * split_df["salary_cap_usd"].astype(float)
        rows.append({
            "split": split,
            "rows": len(split_df),
            "cap_share_mae": mean_absolute_error(y_true, pred_share),
            "usd_mae": mean_absolute_error(split_df["salary_usd"].astype(float), pred_usd),
            "rmse": mean_squared_error(y_true, pred_share) ** 0.5,
            "r2": r2_score(y_true, pred_share),
        })
    return rows


def salary_baseline_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Input: salary modeling data. Output: baseline metrics from train median cap share."""
    train_median = float(df[df["split"].eq("train")][SALARY_TARGET].median())
    rows = []
    for split in SALARY_EVAL_SPLITS:
        split_df = df[df["split"].eq(split)].copy()
        if split_df.empty:
            continue
        pred_share = np.full(len(split_df), train_median)
        pred_usd = pred_share * split_df["salary_cap_usd"].astype(float)
        rows.append({
            "experiment": "salary_cap_share",
            "model": "train_median_baseline",
            "feature_set": "baseline",
            "split": split,
            "rows": len(split_df),
            "features": "train_median_salary_cap_share",
            "cap_share_mae": mean_absolute_error(split_df[SALARY_TARGET].astype(float), pred_share),
            "usd_mae": mean_absolute_error(split_df["salary_usd"].astype(float), pred_usd),
            "rmse": mean_squared_error(split_df[SALARY_TARGET].astype(float), pred_share) ** 0.5,
            "r2": r2_score(split_df[SALARY_TARGET].astype(float), pred_share),
        })
    return pd.DataFrame(rows)


def analyze_salary_features(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Input: salary modeling data and feature list. Output: correlation and mutual-information feature analysis."""
    train_df = df[df["split"].eq("train")].copy()
    numeric_features = [feature for feature in features if pd.api.types.is_numeric_dtype(train_df[feature])]
    rows = []
    for feature in numeric_features:
        pair = pd.concat([pd.to_numeric(train_df[feature], errors="coerce"), train_df[SALARY_TARGET]], axis=1).dropna()
        if len(pair) < 50 or pair.iloc[:, 0].nunique() <= 1:
            continue
        corr = pair.iloc[:, 0].corr(pair.iloc[:, 1])
        rows.append({
            "analysis": "correlation",
            "feature": feature,
            "score": corr,
            "abs_score": abs(corr),
        })

    X_numeric = train_df[numeric_features].copy()
    X_numeric = X_numeric.fillna(X_numeric.median(numeric_only=True))
    mi_scores = mutual_info_regression(X_numeric, train_df[SALARY_TARGET].astype(float), random_state=42)
    rows.extend([
        {
            "analysis": "mutual_info",
            "feature": feature,
            "score": score,
            "abs_score": score,
        }
        for feature, score in zip(numeric_features, mi_scores)
    ])
    return pd.DataFrame(rows).sort_values(["analysis", "abs_score"], ascending=[True, False]).reset_index(drop=True)


def permutation_salary_importance(model: Pipeline, df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Input: fitted salary model and validation data. Output: validation permutation importance by MAE increase."""
    validation_df = df[df["split"].eq("validation")].copy()
    if validation_df.empty:
        return pd.DataFrame(columns=["analysis", "feature", "score", "abs_score", "importance_std"])
    result = permutation_importance(
        model,
        validation_df[features],
        validation_df[SALARY_TARGET].astype(float),
        scoring="neg_mean_absolute_error",
        n_repeats=8,
        random_state=42,
        n_jobs=1,
    )
    return pd.DataFrame({
        "analysis": "permutation_mae_increase",
        "feature": features,
        "score": result.importances_mean,
        "abs_score": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("score", ascending=False).reset_index(drop=True)


def log_salary_model_to_mlflow(
    mlflow,
    model_name: str,
    feature_set_name: str,
    model: Pipeline,
    features: list[str],
    evaluation_df: pd.DataFrame,
    cache_signature: str | None = None,
) -> None:
    """Input: mlflow client, fitted salary model, features, metrics. Output: logged MLflow run."""
    if mlflow is None:
        return
    with mlflow.start_run(run_name=f"salary_{model_name}_{feature_set_name}"):
        params = {
            "target": SALARY_TARGET,
            "feature_set": feature_set_name,
            "n_features": len(features),
            "features": ",".join(features),
            "train_seasons": ",".join(SALARY_TRAIN_SEASONS),
            "validation_seasons": ",".join(SALARY_VALIDATION_SEASONS),
            "test_seasons": ",".join(SALARY_TEST_SEASONS),
        }
        if cache_signature is not None:
            params["settings_signature"] = cache_signature
        mlflow.log_params(params)
        for _, row in evaluation_df.iterrows():
            split = row["split"]
            mlflow.log_metric(f"{split}_rows", int(row["rows"]))
            mlflow.log_metric(f"{split}_cap_share_mae", float(row["cap_share_mae"]))
            mlflow.log_metric(f"{split}_usd_mae", float(row["usd_mae"]))
            mlflow.log_metric(f"{split}_rmse", float(row["rmse"]))
            mlflow.log_metric(f"{split}_r2", float(row["r2"]))
        input_example = salary_modeling[features].head(5)
        mlflow.sklearn.log_model(
            model,
            name="model",
            input_example=input_example,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )


def train_salary_models(df: pd.DataFrame) -> tuple[dict[str, Pipeline], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Input: salary modeling data. Output: fitted models, evaluation, selected summary, and feature importance."""
    train_df = df[df["split"].eq("train")].copy()
    evaluation_rows = []
    models = {}

    baseline_eval = salary_baseline_metrics(df)
    if not baseline_eval.empty:
        evaluation_rows.extend(baseline_eval.to_dict("records"))

    estimators = build_salary_estimators()
    for feature_set_name, raw_features in SALARY_FEATURE_SETS.items():
        features = [feature for feature in raw_features if feature in df.columns]
        for model_name, (estimator, scale_numeric) in estimators.items():
            model = Pipeline([
                ("preprocess", build_salary_preprocessor(df, features, scale_numeric=scale_numeric)),
                ("model", clone(estimator)),
            ])
            model.fit(train_df[features], train_df[SALARY_TARGET].astype(float))
            model_key = f"{model_name}_{feature_set_name}"
            models[model_key] = model
            model_eval = pd.DataFrame(evaluate_salary_model(model, df, features))
            if model_eval.empty:
                continue
            model_eval.insert(0, "experiment", "salary_cap_share")
            model_eval.insert(1, "model", model_name)
            model_eval.insert(2, "feature_set", feature_set_name)
            model_eval.insert(5, "features", ", ".join(features))
            evaluation_rows.extend(model_eval.to_dict("records"))
            log_salary_model_to_mlflow(
                mlflow_client,
                model_name,
                feature_set_name,
                model,
                features,
                model_eval,
                cache_signature=salary_cache_signature,
            )

    evaluation = pd.DataFrame(evaluation_rows)
    validation = evaluation[evaluation["split"].eq("validation")].copy()
    best = validation.sort_values("cap_share_mae").iloc[0]
    test_row = evaluation[
        evaluation["split"].eq("test")
        & evaluation["model"].eq(best["model"])
        & evaluation["feature_set"].eq(best["feature_set"])
    ].head(1)
    selection_summary = pd.DataFrame([{
        "target": SALARY_TARGET,
        "selected_by": "validation_cap_share_mae",
        "recommended_model": best["model"],
        "feature_set": best["feature_set"],
        "validation_cap_share_mae": best["cap_share_mae"],
        "validation_usd_mae": best["usd_mae"],
        "validation_r2": best["r2"],
        "test_cap_share_mae": test_row.iloc[0]["cap_share_mae"] if not test_row.empty else np.nan,
        "test_usd_mae": test_row.iloc[0]["usd_mae"] if not test_row.empty else np.nan,
        "test_r2": test_row.iloc[0]["r2"] if not test_row.empty else np.nan,
    }])

    selected_features = [feature for feature in SALARY_FEATURE_SETS[str(best["feature_set"])] if feature in df.columns]
    selected_model_key = f"{best['model']}_{best['feature_set']}"
    feature_analysis = analyze_salary_features(df, selected_features)
    if selected_model_key in models:
        permutation_analysis = permutation_salary_importance(models[selected_model_key], df, selected_features)
        feature_analysis = pd.concat([feature_analysis, permutation_analysis], ignore_index=True)

    return models, evaluation.sort_values(["split", "cap_share_mae"]).reset_index(drop=True), selection_summary, feature_analysis


salary_modeling = prepare_salary_modeling_data(salary_training_clean)
print("Salary modeling split counts:")
display(salary_modeling.groupby(["season_label", "split"]).size().reset_index(name="rows"))

salary_cache_payload = {
    "cache_version": "salary_cap_share_modern_v1",
    "target": SALARY_TARGET,
    "feature_sets": SALARY_FEATURE_SETS,
    "selected_feature_set": SALARY_SELECTED_FEATURE_SET,
    "selected_model": SALARY_SELECTED_MODEL,
    "train_seasons": SALARY_TRAIN_SEASONS,
    "validation_seasons": SALARY_VALIDATION_SEASONS,
    "test_seasons": SALARY_TEST_SEASONS,
    "estimators": list(build_salary_estimators().keys()),
    "data": dataframe_fingerprint(salary_modeling, split_col="split"),
}
salary_cache_signature = settings_signature(salary_cache_payload)
salary_evaluation = load_metric_cache("salary_evaluation", salary_cache_signature)
salary_selection_summary = load_metric_cache("salary_selection_summary", salary_cache_signature)
salary_feature_importance = load_metric_cache("salary_feature_importance", salary_cache_signature)
if salary_evaluation is not None and salary_selection_summary is not None and salary_feature_importance is not None:
    salary_models = {}
else:
    salary_models, salary_evaluation, salary_selection_summary, salary_feature_importance = train_salary_models(salary_modeling)
    save_metric_cache("salary_evaluation", salary_cache_signature, salary_evaluation, salary_cache_payload)
    save_metric_cache("salary_selection_summary", salary_cache_signature, salary_selection_summary, salary_cache_payload)
    save_metric_cache("salary_feature_importance", salary_cache_signature, salary_feature_importance, salary_cache_payload)

salary_evaluation.to_parquet(SALARY_EVALUATION_PATH, index=False)
salary_selection_summary.to_parquet(SALARY_SELECTION_SUMMARY_PATH, index=False)
salary_feature_importance.to_parquet(SALARY_FEATURE_IMPORTANCE_PATH, index=False)
print(f"Saved {SALARY_EVALUATION_PATH} with shape {salary_evaluation.shape}")
print(f"Saved {SALARY_SELECTION_SUMMARY_PATH} with shape {salary_selection_summary.shape}")
print(f"Saved {SALARY_FEATURE_IMPORTANCE_PATH} with shape {salary_feature_importance.shape}")

display(salary_selection_summary)
display(salary_evaluation.sort_values(["split", "cap_share_mae"]).head(30))
display(salary_feature_importance.groupby("analysis", group_keys=False).head(15))


## 11. Feature Sources

The notebook uses the following feature sources for player replacement search, performance forecasting, and salary analysis:

- Player bio/profile: `data/raw/player_bio/eoin_players.csv` provides `birth_date`, `position`, `height`, and `weight`. Age is calculated relative to each season or anchor date.
- Advanced/rate stats: `data/raw/player_stats_advanced/player_stats_advanced_rs.csv`, `player_stats_usage_rs.csv`, and `player_stats_defense_rs.csv` provide `usage_pct`, `true_shooting_pct`, `steal_rate`, `block_rate`, `defensive_rebound_rate`, `pace`, possessions, and rating fields.
- Recent-season advanced stat coverage: `2023-24` is patched from `rodneycarroll78/nba-stats-1980-2024`; `2024-25` is patched from `ratin21/nba-player-stats-2024-25-per-game`.
- Game context and box-score proxies: `traditional.csv` supports `opponent`, `home_away`, true shooting, rolling form, and defensive per-36 proxies.
- Salary market context: `data/raw/salary_cap/salary_cap_by_season.csv` supports `salary_cap_share`.

## 12. Project Roadmap

Near-term modeling and data priorities:

- Validate salary joins using both same-season descriptive analysis and prior-season predictive features.
- Refine role dimensions for replacement-player search and similarity explanations.
- Compare short-term horizons such as next five games, next ten games, and remainder-of-season production.
- Expand model diagnostics with feature importance, calibration, residual analysis, and target-specific error slices.
- Promote stable notebook functions into tested data-processing modules after the exploratory pipeline is finalized.